Generate Study 1 LNPDB simulated dataset (N=19,200) with realistic physicochemical features

In [1]:

import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder

np.random.seed(42)
N = 19200

# --- Lipid types (ionizable lipids used in LNP literature) ---
ionizable_lipids = ['MC3', 'DLin-MC3-DMA', 'C12-200', 'Lipid5', 'SM-102',
                    'ALC-0315', 'OF-02', 'cKK-E12', 'DODAP', 'DOTAP',
                    'L319', 'A2-Iso5-2DC18', 'FTT5', 'BAMEA-O16B', 'Lipid9']
helper_lipids = ['DSPC', 'DOPE', 'DPPC', 'DOPC', 'DSPE']
peg_lipids = ['DMG-PEG2000', 'DSPE-PEG2000', 'C14-PEG2000']
cell_lines = ['HeLa', 'HepG2', 'A549', 'Huh7', 'NIH3T3', 'RAW264.7', 'HUVEC', 'MCF7']
cargo_types = ['mRNA', 'siRNA', 'pDNA', 'saRNA']
sources = ['Kulkarni2021', 'Hassett2019', 'Sabnis2018', 'Jayaraman2012',
           'Semple2010', 'Maier2013', 'Patel2017', 'Billingsley2020',
           'Fenton2016', 'Miao2020', 'Moderna2021', 'BioNTech2021',
           'Rosenblum2020', 'Kauffman2015', 'Dahlman2017']

# --- Molar ratios (must sum to ~100) ---
# CHL: 30-50, HL: 5-20, PEG: 1-5, ionizable: remainder
CHL_mol = np.random.normal(40, 6, N).clip(25, 55)
HL_mol  = np.random.normal(10, 3, N).clip(3, 20)
PEG_mol = np.random.normal(2.5, 0.8, N).clip(0.5, 5)
IL_mol  = 100 - CHL_mol - HL_mol - PEG_mol
IL_mol  = IL_mol.clip(30, 60)

# Normalize to sum=100
total = CHL_mol + HL_mol + PEG_mol + IL_mol
CHL_mol = CHL_mol / total * 100
HL_mol  = HL_mol  / total * 100
PEG_mol = PEG_mol / total * 100
IL_mol  = IL_mol  / total * 100

# --- Physicochemical properties ---
size_nm       = np.random.lognormal(np.log(100), 0.3, N).clip(50, 300)
PDI           = np.random.beta(2, 8, N).clip(0.05, 0.4)
zeta_mV       = np.random.normal(-5, 8, N).clip(-40, 30)
encap_eff     = np.random.beta(8, 2, N).clip(0.5, 1.0) * 100
N_P_ratio     = np.random.lognormal(np.log(6), 0.4, N).clip(1, 20)

# --- Molecular descriptors of ionizable lipid ---
Frac_sp3C     = np.random.beta(5, 3, N).clip(0.3, 0.95)
MW            = np.random.normal(650, 80, N).clip(400, 900)
logP          = np.random.normal(8, 1.5, N).clip(4, 14)
HBD           = np.random.randint(0, 4, N).astype(float)
HBA           = np.random.randint(2, 8, N).astype(float)
pKa           = np.random.normal(6.5, 0.8, N).clip(4.5, 9.0)
TPSA          = np.random.normal(60, 15, N).clip(20, 120)
RotBonds      = np.random.randint(5, 20, N).astype(float)
AromaticRings = np.random.randint(0, 3, N).astype(float)

# --- Categorical ---
ionizable_lipid_col = np.random.choice(ionizable_lipids, N)
helper_lipid_col    = np.random.choice(helper_lipids, N)
peg_lipid_col       = np.random.choice(peg_lipids, N)
cell_line_col       = np.random.choice(cell_lines, N)
cargo_col           = np.random.choice(cargo_types, N, p=[0.45, 0.30, 0.15, 0.10])
source_col          = np.random.choice(sources, N)

# --- Efficacy label (binary: high/low transfection) ---
# Driven by: CHL in 35-45%, HL 8-14%, PEG <3%, pKa 6-7, PDI<0.2, size 80-120
score = (
    - 0.5 * np.abs(CHL_mol - 40) / 5
    - 0.4 * np.abs(HL_mol - 11) / 3
    - 0.6 * (PEG_mol - 2.5).clip(0) / 2
    - 0.3 * np.abs(pKa - 6.5) / 0.8
    - 0.4 * PDI / 0.2
    - 0.2 * np.abs(np.log(size_nm) - np.log(100)) / 0.3
    + 0.3 * Frac_sp3C
    + 0.2 * (encap_eff - 70) / 20
    + np.random.normal(0, 0.5, N)
)
prob = 1 / (1 + np.exp(-score))
efficacy_binary = (prob > 0.5).astype(int)

# Efficacy class (4-class)
efficacy_class = pd.cut(prob,
    bins=[0, 0.25, 0.50, 0.75, 1.0],
    labels=['Very Low', 'Low', 'High', 'Very High'])

# --- Assemble DataFrame ---
df = pd.DataFrame({
    'record_id':        [f'LNPDB_{i:05d}' for i in range(N)],
    'source':           source_col,
    'ionizable_lipid':  ionizable_lipid_col,
    'helper_lipid':     helper_lipid_col,
    'peg_lipid':        peg_lipid_col,
    'cell_line':        cell_line_col,
    'cargo_type':       cargo_col,
    'CHL_mol_pct':      np.round(CHL_mol, 2),
    'HL_mol_pct':       np.round(HL_mol, 2),
    'PEG_mol_pct':      np.round(PEG_mol, 2),
    'IL_mol_pct':       np.round(IL_mol, 2),
    'size_nm':          np.round(size_nm, 1),
    'PDI':              np.round(PDI, 3),
    'zeta_mV':          np.round(zeta_mV, 1),
    'encap_eff_pct':    np.round(encap_eff, 1),
    'N_P_ratio':        np.round(N_P_ratio, 2),
    'Frac_sp3C':        np.round(Frac_sp3C, 3),
    'MW_Da':            np.round(MW, 1),
    'logP':             np.round(logP, 2),
    'HBD':              HBD.astype(int),
    'HBA':              HBA.astype(int),
    'pKa':              np.round(pKa, 2),
    'TPSA':             np.round(TPSA, 1),
    'RotBonds':         RotBonds.astype(int),
    'AromaticRings':    AromaticRings.astype(int),
    'efficacy_binary':  efficacy_binary,
    'efficacy_class':   efficacy_class,
    'transfection_prob': np.round(prob, 4),
})

df.to_csv('/mnt/results/study1/data/LNPDB_SIMULATED_N19200.csv', index=False)
print(f"Dataset shape: {df.shape}")
print(f"\nEfficacy class distribution:\n{df['efficacy_class'].value_counts()}")
print(f"\nBinary efficacy: {df['efficacy_binary'].value_counts().to_dict()}")
print(f"\nCargo distribution:\n{df['cargo_type'].value_counts()}")
print(f"\nSource distribution:\n{df['source'].value_counts()}")
print(f"\nFeature stats (molar ratios):")
print(df[['CHL_mol_pct','HL_mol_pct','PEG_mol_pct','IL_mol_pct']].describe().round(2))


Dataset shape: (19200, 28)

Efficacy class distribution:
efficacy_class
Very Low     12500
Low           6213
High           485
Very High        2
Name: count, dtype: int64

Binary efficacy: {0: 18713, 1: 487}

Cargo distribution:
cargo_type
mRNA     8706
siRNA    5749
pDNA     2857
saRNA    1888
Name: count, dtype: int64

Source distribution:
source
Semple2010         1329
Jayaraman2012      1304
Maier2013          1303
Fenton2016         1300
Kulkarni2021       1299
Billingsley2020    1292
Kauffman2015       1287
Sabnis2018         1281
Miao2020           1271
BioNTech2021       1271
Patel2017          1269
Moderna2021        1258
Hassett2019        1254
Dahlman2017        1251
Rosenblum2020      1231
Name: count, dtype: int64

Feature stats (molar ratios):
       CHL_mol_pct  HL_mol_pct  PEG_mol_pct  IL_mol_pct
count     19200.00    19200.00     19200.00    19200.00
mean         40.05        9.97         2.51       47.48
std           5.89        2.96         0.80        6.59
min  

Regenerate LNPDB dataset with realistic class balance (~40% high efficacy)

In [3]:

import numpy as np
import pandas as pd

np.random.seed(42)
N = 19200

ionizable_lipids = ['MC3', 'DLin-MC3-DMA', 'C12-200', 'Lipid5', 'SM-102',
                    'ALC-0315', 'OF-02', 'cKK-E12', 'DODAP', 'DOTAP',
                    'L319', 'A2-Iso5-2DC18', 'FTT5', 'BAMEA-O16B', 'Lipid9']
helper_lipids = ['DSPC', 'DOPE', 'DPPC', 'DOPC', 'DSPE']
peg_lipids    = ['DMG-PEG2000', 'DSPE-PEG2000', 'C14-PEG2000']
cell_lines    = ['HeLa', 'HepG2', 'A549', 'Huh7', 'NIH3T3', 'RAW264.7', 'HUVEC', 'MCF7']
cargo_types   = ['mRNA', 'siRNA', 'pDNA', 'saRNA']
sources       = ['Kulkarni2021', 'Hassett2019', 'Sabnis2018', 'Jayaraman2012',
                 'Semple2010', 'Maier2013', 'Patel2017', 'Billingsley2020',
                 'Fenton2016', 'Miao2020', 'Moderna2021', 'BioNTech2021',
                 'Rosenblum2020', 'Kauffman2015', 'Dahlman2017']

# Molar ratios
CHL_mol = np.random.normal(40, 6, N).clip(25, 55)
HL_mol  = np.random.normal(10, 3, N).clip(3, 20)
PEG_mol = np.random.normal(2.5, 0.8, N).clip(0.5, 5)
IL_mol  = (100 - CHL_mol - HL_mol - PEG_mol).clip(30, 60)
total   = CHL_mol + HL_mol + PEG_mol + IL_mol
CHL_mol = CHL_mol / total * 100
HL_mol  = HL_mol  / total * 100
PEG_mol = PEG_mol / total * 100
IL_mol  = IL_mol  / total * 100

# Physicochemical
size_nm      = np.random.lognormal(np.log(100), 0.3, N).clip(50, 300)
PDI          = np.random.beta(2, 8, N).clip(0.05, 0.4)
zeta_mV      = np.random.normal(-5, 8, N).clip(-40, 30)
encap_eff    = np.random.beta(8, 2, N).clip(0.5, 1.0) * 100
N_P_ratio    = np.random.lognormal(np.log(6), 0.4, N).clip(1, 20)
Frac_sp3C    = np.random.beta(5, 3, N).clip(0.3, 0.95)
MW           = np.random.normal(650, 80, N).clip(400, 900)
logP         = np.random.normal(8, 1.5, N).clip(4, 14)
HBD          = np.random.randint(0, 4, N).astype(float)
HBA          = np.random.randint(2, 8, N).astype(float)
pKa          = np.random.normal(6.5, 0.8, N).clip(4.5, 9.0)
TPSA         = np.random.normal(60, 15, N).clip(20, 120)
RotBonds     = np.random.randint(5, 20, N).astype(float)
AromaticRings= np.random.randint(0, 3, N).astype(float)

# Efficacy score — calibrated so ~40% are "high efficacy"
# Key drivers: CHL near 40%, HL near 10-12%, PEG low, pKa 6-7, small PDI
score = (
    - 0.8 * (np.abs(CHL_mol - 40) / 5) ** 1.2
    - 0.6 * (np.abs(HL_mol - 11) / 3) ** 1.2
    - 1.0 * (PEG_mol - 2.5).clip(0) / 1.5
    - 0.5 * np.abs(pKa - 6.5) / 0.8
    - 0.8 * PDI / 0.15
    - 0.3 * np.abs(np.log(size_nm) - np.log(100)) / 0.3
    + 0.5 * Frac_sp3C
    + 0.3 * (encap_eff - 70) / 20
    + np.random.normal(0, 0.8, N)
)

# Shift intercept so ~40% positive
threshold = np.percentile(score, 60)   # top 40% = high efficacy
efficacy_binary = (score >= threshold).astype(int)

# Continuous probability for ROC/class labels
prob = 1 / (1 + np.exp(-(score - threshold)))

efficacy_class = pd.cut(
    prob,
    bins=[0, 0.25, 0.50, 0.75, 1.0],
    labels=['Very Low', 'Low', 'High', 'Very High'],
    include_lowest=True
)

# Categoricals
ionizable_lipid_col = np.random.choice(ionizable_lipids, N)
helper_lipid_col    = np.random.choice(helper_lipids, N)
peg_lipid_col       = np.random.choice(peg_lipids, N)
cell_line_col       = np.random.choice(cell_lines, N)
cargo_col           = np.random.choice(cargo_types, N, p=[0.45, 0.30, 0.15, 0.10])
source_col          = np.random.choice(sources, N)

df = pd.DataFrame({
    'record_id':         [f'LNPDB_{i:05d}' for i in range(N)],
    'source':            source_col,
    'ionizable_lipid':   ionizable_lipid_col,
    'helper_lipid':      helper_lipid_col,
    'peg_lipid':         peg_lipid_col,
    'cell_line':         cell_line_col,
    'cargo_type':        cargo_col,
    'CHL_mol_pct':       np.round(CHL_mol, 2),
    'HL_mol_pct':        np.round(HL_mol, 2),
    'PEG_mol_pct':       np.round(PEG_mol, 2),
    'IL_mol_pct':        np.round(IL_mol, 2),
    'size_nm':           np.round(size_nm, 1),
    'PDI':               np.round(PDI, 3),
    'zeta_mV':           np.round(zeta_mV, 1),
    'encap_eff_pct':     np.round(encap_eff, 1),
    'N_P_ratio':         np.round(N_P_ratio, 2),
    'Frac_sp3C':         np.round(Frac_sp3C, 3),
    'MW_Da':             np.round(MW, 1),
    'logP':              np.round(logP, 2),
    'HBD':               HBD.astype(int),
    'HBA':               HBA.astype(int),
    'pKa':               np.round(pKa, 2),
    'TPSA':              np.round(TPSA, 1),
    'RotBonds':          RotBonds.astype(int),
    'AromaticRings':     AromaticRings.astype(int),
    'efficacy_binary':   efficacy_binary,
    'efficacy_class':    efficacy_class,
    'transfection_prob': np.round(prob, 4),
})

df.to_csv('/mnt/results/study1/data/LNPDB_SIMULATED_N19200.csv', index=False)
print(f"Dataset shape: {df.shape}")
print(f"\nEfficacy class distribution:\n{df['efficacy_class'].value_counts().sort_index()}")
print(f"\nBinary efficacy: {df['efficacy_binary'].value_counts().to_dict()}")
print(f"\nNo missing values: {df.isnull().sum().sum() == 0}")
print(f"\nDuplicate record_ids: {df['record_id'].duplicated().sum()}")


Dataset shape: (19200, 28)

Efficacy class distribution:
efficacy_class
Very Low     5716
Low          5804
High         5099
Very High    2581
Name: count, dtype: int64

Binary efficacy: {0: 11520, 1: 7680}

No missing values: True

Duplicate record_ids: 0


Study 1 Figure 1 — Data overview: source distribution + feature distributions

In [5]:

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec

df = pd.read_csv('/mnt/results/study1/data/LNPDB_SIMULATED_N19200.csv')

# ColorBrewer Set2 palette
CB = ['#66c2a5','#fc8d62','#8da0cb','#e78ac3','#a6d854','#ffd92f','#e5c494','#b3b3b3']

fig = plt.figure(figsize=(14, 10), facecolor='white')
fig.suptitle('Figure 1 — LNPDB Dataset Overview\n(SIMULATED DATA, N = 19,200)',
             fontsize=14, fontweight='bold', y=0.98)

gs = GridSpec(3, 3, figure=fig, hspace=0.55, wspace=0.40)

# --- Panel A: Source distribution ---
ax0 = fig.add_subplot(gs[0, :2])
src_counts = df['source'].value_counts().sort_values(ascending=True)
bars = ax0.barh(src_counts.index, src_counts.values, color=CB[0], edgecolor='white', linewidth=0.5)
ax0.set_xlabel('Number of Records', fontsize=10)
ax0.set_title('A  Records per Literature Source', fontsize=11, fontweight='bold', loc='left')
ax0.tick_params(axis='y', labelsize=8)
for bar, val in zip(bars, src_counts.values):
    ax0.text(val + 10, bar.get_y() + bar.get_height()/2, str(val),
             va='center', fontsize=7, color='#333333')
ax0.set_xlim(0, src_counts.max() * 1.15)
ax0.spines[['top','right']].set_visible(False)

# --- Panel B: Cargo type pie ---
ax1 = fig.add_subplot(gs[0, 2])
cargo_counts = df['cargo_type'].value_counts()
wedges, texts, autotexts = ax1.pie(
    cargo_counts.values, labels=cargo_counts.index,
    autopct='%1.1f%%', colors=CB[:4],
    startangle=90, textprops={'fontsize': 8})
for at in autotexts:
    at.set_fontsize(7)
ax1.set_title('B  Cargo Type', fontsize=11, fontweight='bold', loc='left')

# --- Panel C: CHL mol% histogram ---
ax2 = fig.add_subplot(gs[1, 0])
ax2.hist(df['CHL_mol_pct'], bins=40, color=CB[0], edgecolor='white', linewidth=0.3)
ax2.set_xlabel('CHL mol%', fontsize=9)
ax2.set_ylabel('Count', fontsize=9)
ax2.set_title('C  Cholesterol mol%', fontsize=10, fontweight='bold', loc='left')
ax2.spines[['top','right']].set_visible(False)
ax2.tick_params(labelsize=8)

# --- Panel D: HL mol% histogram ---
ax3 = fig.add_subplot(gs[1, 1])
ax3.hist(df['HL_mol_pct'], bins=40, color=CB[1], edgecolor='white', linewidth=0.3)
ax3.set_xlabel('HL mol%', fontsize=9)
ax3.set_ylabel('Count', fontsize=9)
ax3.set_title('D  Helper Lipid mol%', fontsize=10, fontweight='bold', loc='left')
ax3.spines[['top','right']].set_visible(False)
ax3.tick_params(labelsize=8)

# --- Panel E: PEG mol% histogram ---
ax4 = fig.add_subplot(gs[1, 2])
ax4.hist(df['PEG_mol_pct'], bins=40, color=CB[2], edgecolor='white', linewidth=0.3)
ax4.set_xlabel('PEG mol%', fontsize=9)
ax4.set_ylabel('Count', fontsize=9)
ax4.set_title('E  PEG mol%', fontsize=10, fontweight='bold', loc='left')
ax4.spines[['top','right']].set_visible(False)
ax4.tick_params(labelsize=8)

# --- Panel F: Frac.sp3C histogram ---
ax5 = fig.add_subplot(gs[2, 0])
ax5.hist(df['Frac_sp3C'], bins=40, color=CB[3], edgecolor='white', linewidth=0.3)
ax5.set_xlabel('Frac. sp³C', fontsize=9)
ax5.set_ylabel('Count', fontsize=9)
ax5.set_title('F  Frac. sp³C (ionizable lipid)', fontsize=10, fontweight='bold', loc='left')
ax5.spines[['top','right']].set_visible(False)
ax5.tick_params(labelsize=8)

# --- Panel G: Size distribution ---
ax6 = fig.add_subplot(gs[2, 1])
ax6.hist(df['size_nm'], bins=40, color=CB[4], edgecolor='white', linewidth=0.3)
ax6.set_xlabel('Particle Size (nm)', fontsize=9)
ax6.set_ylabel('Count', fontsize=9)
ax6.set_title('G  Particle Size', fontsize=10, fontweight='bold', loc='left')
ax6.spines[['top','right']].set_visible(False)
ax6.tick_params(labelsize=8)

# --- Panel H: Efficacy class bar ---
ax7 = fig.add_subplot(gs[2, 2])
eff_counts = df['efficacy_class'].value_counts().reindex(['Very Low','Low','High','Very High'])
ax7.bar(eff_counts.index, eff_counts.values, color=CB[:4], edgecolor='white', linewidth=0.5)
ax7.set_xlabel('Efficacy Class', fontsize=9)
ax7.set_ylabel('Count', fontsize=9)
ax7.set_title('H  Efficacy Class', fontsize=10, fontweight='bold', loc='left')
ax7.tick_params(axis='x', labelsize=7, rotation=20)
ax7.spines[['top','right']].set_visible(False)
ax7.tick_params(labelsize=8)

# Watermark note
fig.text(0.99, 0.01, 'SIMULATED DATA — for methodology demonstration only',
         ha='right', va='bottom', fontsize=7, color='#888888', style='italic')

plt.savefig('/mnt/results/study1/figures/Figure1.png', dpi=300, bbox_inches='tight',
            facecolor='white')
plt.savefig('/mnt/results/study1/figures/Figure1.svg', bbox_inches='tight',
            facecolor='white')
plt.close()
print("Figure1 saved.")


Figure1 saved.


Study 1 Figure 2 — Correlation heatmap with hierarchical clustering

In [7]:

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.cluster.hierarchy import linkage, dendrogram, leaves_list
from scipy.spatial.distance import squareform

df = pd.read_csv('/mnt/results/study1/data/LNPDB_SIMULATED_N19200.csv')

num_cols = ['CHL_mol_pct','HL_mol_pct','PEG_mol_pct','IL_mol_pct',
            'size_nm','PDI','zeta_mV','encap_eff_pct','N_P_ratio',
            'Frac_sp3C','MW_Da','logP','HBD','HBA','pKa','TPSA',
            'RotBonds','AromaticRings','efficacy_binary']

nice_labels = {
    'CHL_mol_pct':   'CHL mol%',
    'HL_mol_pct':    'HL mol%',
    'PEG_mol_pct':   'PEG mol%',
    'IL_mol_pct':    'IL mol%',
    'size_nm':       'Size (nm)',
    'PDI':           'PDI',
    'zeta_mV':       'Zeta (mV)',
    'encap_eff_pct': 'Encap. Eff. (%)',
    'N_P_ratio':     'N/P Ratio',
    'Frac_sp3C':     'Frac. sp³C',
    'MW_Da':         'MW (Da)',
    'logP':          'logP',
    'HBD':           'HBD',
    'HBA':           'HBA',
    'pKa':           'pKa',
    'TPSA':          'TPSA',
    'RotBonds':      'Rot. Bonds',
    'AromaticRings': 'Arom. Rings',
    'efficacy_binary':'Efficacy'
}

corr = df[num_cols].corr()
corr.index   = [nice_labels[c] for c in corr.index]
corr.columns = [nice_labels[c] for c in corr.columns]

# Hierarchical clustering
dist = 1 - np.abs(corr.values)
np.fill_diagonal(dist, 0)
dist = np.clip(dist, 0, None)
Z = linkage(squareform(dist), method='average')
order = leaves_list(Z)
corr_ordered = corr.iloc[order, order]

fig, ax = plt.subplots(figsize=(11, 9), facecolor='white')
mask = np.zeros_like(corr_ordered, dtype=bool)

# RdBu_r is colorblind-friendly diverging
sns.heatmap(
    corr_ordered, ax=ax,
    cmap='RdBu_r', vmin=-1, vmax=1,
    annot=True, fmt='.2f', annot_kws={'size': 6.5},
    linewidths=0.4, linecolor='#dddddd',
    cbar_kws={'label': 'Pearson r', 'shrink': 0.75}
)

ax.set_title('Figure 2 — Feature Correlation Matrix (Hierarchical Clustering)\n(SIMULATED DATA, N = 19,200)',
             fontsize=12, fontweight='bold', pad=14)
ax.tick_params(axis='x', labelsize=8, rotation=45)
ax.tick_params(axis='y', labelsize=8, rotation=0)

fig.text(0.99, 0.01, 'SIMULATED DATA — for methodology demonstration only',
         ha='right', va='bottom', fontsize=7, color='#888888', style='italic')

plt.tight_layout()
plt.savefig('/mnt/results/study1/figures/Figure2.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.savefig('/mnt/results/study1/figures/Figure2.svg', bbox_inches='tight', facecolor='white')
plt.close()
print("Figure2 saved.")


Figure2 saved.


Train XGBoost model on LNPDB simulated data, verify AUC≈0.791, compute SHAP values

In [9]:

import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix, classification_report
import xgboost as xgb
import shap
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('/mnt/results/study1/data/LNPDB_SIMULATED_N19200.csv')

# Feature columns
feat_cols = ['CHL_mol_pct','HL_mol_pct','PEG_mol_pct','IL_mol_pct',
             'size_nm','PDI','zeta_mV','encap_eff_pct','N_P_ratio',
             'Frac_sp3C','MW_Da','logP','HBD','HBA','pKa','TPSA',
             'RotBonds','AromaticRings']

X = df[feat_cols].values
y = df['efficacy_binary'].values

# XGBoost with params tuned to land near AUC=0.791
model = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=5,
    gamma=1.0,
    reg_alpha=0.5,
    reg_lambda=2.0,
    scale_pos_weight=1,
    use_label_encoder=False,
    eval_metric='logloss',
    random_state=42,
    n_jobs=-1
)

# 5-fold stratified CV for unbiased predictions
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
y_prob_cv = cross_val_predict(model, X, y, cv=cv, method='predict_proba')[:, 1]
y_pred_cv = (y_prob_cv >= 0.5).astype(int)

auc_cv = roc_auc_score(y, y_prob_cv)
print(f"5-fold CV AUC: {auc_cv:.4f}")
print(f"\nClassification Report:\n{classification_report(y, y_pred_cv)}")

# Fit on full data for SHAP
model.fit(X, y)

# SHAP values
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X)

# Mean |SHAP| per feature
mean_shap = np.abs(shap_values).mean(axis=0)
shap_df = pd.DataFrame({
    'feature': feat_cols,
    'mean_abs_shap': mean_shap
}).sort_values('mean_abs_shap', ascending=False)

# Normalize to sum=1 for display
shap_df['shap_norm'] = shap_df['mean_abs_shap'] / shap_df['mean_abs_shap'].sum()
print(f"\nTop 10 SHAP features (normalized):")
print(shap_df.head(10).to_string(index=False))

# Save CV predictions and SHAP
cv_results = pd.DataFrame({
    'record_id': df['record_id'],
    'y_true': y,
    'y_prob': np.round(y_prob_cv, 4),
    'y_pred': y_pred_cv,
    'efficacy_class': df['efficacy_class']
})
cv_results.to_csv('/mnt/results/study1/data/LNPDB_SIMULATED_cv_predictions.csv', index=False)
shap_df.to_csv('/mnt/results/study1/data/LNPDB_SIMULATED_shap_values.csv', index=False)

# Save full SHAP matrix for beeswarm
np.save('/mnt/results/study1/data/shap_matrix.npy', shap_values)
np.save('/mnt/results/study1/data/X_features.npy', X)

print(f"\nAUC = {auc_cv:.3f} (target: ~0.791)")


Traceback (most recent call last):
  File "<string>", line 71, in <module>
  File "<string>", line 7, in <module>
ModuleNotFoundError: No module named 'xgboost'


Train XGBoost, 5-fold CV AUC, compute SHAP values on LNPDB simulated data

In [11]:

import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import roc_auc_score, classification_report
import xgboost as xgb
import shap
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('/mnt/results/study1/data/LNPDB_SIMULATED_N19200.csv')

feat_cols = ['CHL_mol_pct','HL_mol_pct','PEG_mol_pct','IL_mol_pct',
             'size_nm','PDI','zeta_mV','encap_eff_pct','N_P_ratio',
             'Frac_sp3C','MW_Da','logP','HBD','HBA','pKa','TPSA',
             'RotBonds','AromaticRings']

X = df[feat_cols].values
y = df['efficacy_binary'].values

model = xgb.XGBClassifier(
    n_estimators=300, max_depth=5, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    min_child_weight=5, gamma=1.0,
    reg_alpha=0.5, reg_lambda=2.0,
    eval_metric='logloss', random_state=42, n_jobs=-1
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
y_prob_cv = cross_val_predict(model, X, y, cv=cv, method='predict_proba')[:, 1]
y_pred_cv = (y_prob_cv >= 0.5).astype(int)
auc_cv = roc_auc_score(y, y_prob_cv)
print(f"5-fold CV AUC: {auc_cv:.4f}")
print(f"\nClassification Report:\n{classification_report(y, y_pred_cv)}")

# Fit full model for SHAP
model.fit(X, y)
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X)

mean_shap = np.abs(shap_values).mean(axis=0)
shap_df = pd.DataFrame({'feature': feat_cols, 'mean_abs_shap': mean_shap})
shap_df['shap_norm'] = shap_df['mean_abs_shap'] / shap_df['mean_abs_shap'].sum()
shap_df = shap_df.sort_values('mean_abs_shap', ascending=False).reset_index(drop=True)

print(f"\nTop 10 SHAP features (normalized):")
print(shap_df.head(10)[['feature','shap_norm']].to_string(index=False))

# Save outputs
cv_results = pd.DataFrame({
    'record_id': df['record_id'], 'y_true': y,
    'y_prob': np.round(y_prob_cv, 4), 'y_pred': y_pred_cv,
    'efficacy_class': df['efficacy_class']
})
cv_results.to_csv('/mnt/results/study1/data/LNPDB_SIMULATED_cv_predictions.csv', index=False)
shap_df.to_csv('/mnt/results/study1/data/LNPDB_SIMULATED_shap_values.csv', index=False)
np.save('/mnt/results/study1/data/shap_matrix.npy', shap_values)
np.save('/mnt/results/study1/data/X_features.npy', X)
print(f"\nAll outputs saved. AUC = {auc_cv:.3f}")


5-fold CV AUC: 0.8770

Classification Report:
              precision    recall  f1-score   support

           0       0.82      0.84      0.83     11520
           1       0.75      0.73      0.74      7680

    accuracy                           0.80     19200
   macro avg       0.79      0.79      0.79     19200
weighted avg       0.80      0.80      0.80     19200


Top 10 SHAP features (normalized):
      feature  shap_norm
          PDI   0.210022
  CHL_mol_pct   0.191712
   HL_mol_pct   0.133673
  PEG_mol_pct   0.104683
          pKa   0.099120
      size_nm   0.068954
encap_eff_pct   0.059404
   IL_mol_pct   0.054707
    Frac_sp3C   0.028970
         logP   0.010710

All outputs saved. AUC = 0.877


Study 1 Figure 3 — ROC curves for all efficacy classes (OvR)

In [13]:

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize
from sklearn.model_selection import StratifiedKFold, cross_val_predict
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

df   = pd.read_csv('/mnt/results/study1/data/LNPDB_SIMULATED_N19200.csv')
cv_r = pd.read_csv('/mnt/results/study1/data/LNPDB_SIMULATED_cv_predictions.csv')

# ColorBrewer Set1 (colorblind-safe for lines)
CB = ['#e41a1c','#377eb8','#4daf4a','#984ea3','#ff7f00']

feat_cols = ['CHL_mol_pct','HL_mol_pct','PEG_mol_pct','IL_mol_pct',
             'size_nm','PDI','zeta_mV','encap_eff_pct','N_P_ratio',
             'Frac_sp3C','MW_Da','logP','HBD','HBA','pKa','TPSA',
             'RotBonds','AromaticRings']
X = df[feat_cols].values
classes = ['Very Low','Low','High','Very High']
y_multi = df['efficacy_class'].values

# Encode
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
le.fit(classes)
y_enc = le.transform(y_multi)
y_bin = label_binarize(y_enc, classes=[0,1,2,3])

# Multiclass XGB CV probabilities
model_mc = xgb.XGBClassifier(
    n_estimators=300, max_depth=5, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    min_child_weight=5, gamma=1.0,
    reg_alpha=0.5, reg_lambda=2.0,
    objective='multi:softprob', num_class=4,
    eval_metric='mlogloss', random_state=42, n_jobs=-1
)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
y_prob_mc = cross_val_predict(model_mc, X, y_enc, cv=cv, method='predict_proba')

# --- Plot ---
fig, axes = plt.subplots(1, 2, figsize=(13, 6), facecolor='white')

# Left: per-class OvR ROC
ax = axes[0]
aucs = []
for i, cls in enumerate(classes):
    fpr, tpr, _ = roc_curve(y_bin[:, i], y_prob_mc[:, i])
    roc_auc = auc(fpr, tpr)
    aucs.append(roc_auc)
    ax.plot(fpr, tpr, color=CB[i], lw=2,
            label=f'{cls} (AUC = {roc_auc:.3f})')

ax.plot([0,1],[0,1], 'k--', lw=1, alpha=0.5, label='Random (AUC = 0.500)')
ax.set_xlabel('False Positive Rate', fontsize=11)
ax.set_ylabel('True Positive Rate', fontsize=11)
ax.set_title('A  Per-Class ROC (One-vs-Rest)', fontsize=12, fontweight='bold', loc='left')
ax.legend(fontsize=9, loc='lower right')
ax.set_xlim([0,1]); ax.set_ylim([0,1.02])
ax.spines[['top','right']].set_visible(False)
ax.tick_params(labelsize=9)

# Right: binary ROC
ax2 = axes[1]
y_bin_true = df['efficacy_binary'].values
y_bin_prob = cv_r['y_prob'].values
fpr_b, tpr_b, _ = roc_curve(y_bin_true, y_bin_prob)
auc_b = auc(fpr_b, tpr_b)
ax2.plot(fpr_b, tpr_b, color=CB[1], lw=2.5,
         label=f'Binary classifier (AUC = {auc_b:.3f})')
ax2.fill_between(fpr_b, tpr_b, alpha=0.12, color=CB[1])
ax2.plot([0,1],[0,1], 'k--', lw=1, alpha=0.5, label='Random (AUC = 0.500)')
ax2.set_xlabel('False Positive Rate', fontsize=11)
ax2.set_ylabel('True Positive Rate', fontsize=11)
ax2.set_title('B  Binary ROC (High vs Low Efficacy)', fontsize=12, fontweight='bold', loc='left')
ax2.legend(fontsize=10, loc='lower right')
ax2.set_xlim([0,1]); ax2.set_ylim([0,1.02])
ax2.spines[['top','right']].set_visible(False)
ax2.tick_params(labelsize=9)

fig.suptitle('Figure 3 — ROC Curves: XGBoost Transfection Efficacy Classifier\n(SIMULATED DATA, N = 19,200 | 5-fold CV)',
             fontsize=13, fontweight='bold', y=1.01)

fig.text(0.99, 0.01, 'SIMULATED DATA — for methodology demonstration only',
         ha='right', va='bottom', fontsize=7, color='#888888', style='italic')

plt.tight_layout()
plt.savefig('/mnt/results/study1/figures/Figure3.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.savefig('/mnt/results/study1/figures/Figure3.svg', bbox_inches='tight', facecolor='white')
plt.close()
print(f"Figure3 saved. Binary AUC={auc_b:.3f}, Class AUCs={[f'{a:.3f}' for a in aucs]}")


Figure3 saved. Binary AUC=0.877, Class AUCs=['0.745', '0.689', '0.890', '0.896']


Study 1 Figure 4 — SHAP beeswarm plot (top 15 features)

In [15]:

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from matplotlib.colors import Normalize

df       = pd.read_csv('/mnt/results/study1/data/LNPDB_SIMULATED_N19200.csv')
shap_df  = pd.read_csv('/mnt/results/study1/data/LNPDB_SIMULATED_shap_values.csv')
shap_mat = np.load('/mnt/results/study1/data/shap_matrix.npy')
X        = np.load('/mnt/results/study1/data/X_features.npy')

feat_cols = ['CHL_mol_pct','HL_mol_pct','PEG_mol_pct','IL_mol_pct',
             'size_nm','PDI','zeta_mV','encap_eff_pct','N_P_ratio',
             'Frac_sp3C','MW_Da','logP','HBD','HBA','pKa','TPSA',
             'RotBonds','AromaticRings']

nice_labels = {
    'CHL_mol_pct':   'CHL mol%',
    'HL_mol_pct':    'HL mol%',
    'PEG_mol_pct':   'PEG mol%',
    'IL_mol_pct':    'IL mol%',
    'size_nm':       'Size (nm)',
    'PDI':           'PDI',
    'zeta_mV':       'Zeta (mV)',
    'encap_eff_pct': 'Encap. Eff. (%)',
    'N_P_ratio':     'N/P Ratio',
    'Frac_sp3C':     'Frac. sp³C',
    'MW_Da':         'MW (Da)',
    'logP':          'logP',
    'HBD':           'HBD',
    'HBA':           'HBA',
    'pKa':           'pKa',
    'TPSA':          'TPSA',
    'RotBonds':      'Rot. Bonds',
    'AromaticRings': 'Arom. Rings'
}

# Top 15 features by mean |SHAP|
top15 = shap_df.head(15)['feature'].tolist()
top15_idx = [feat_cols.index(f) for f in top15]

# Order bottom-to-top (highest SHAP at top)
top15_rev = top15[::-1]
top15_idx_rev = [feat_cols.index(f) for f in top15_rev]

# Subsample for speed/clarity
rng = np.random.default_rng(42)
idx_sample = rng.choice(len(X), size=3000, replace=False)
X_s    = X[idx_sample]
shap_s = shap_mat[idx_sample]

fig, ax = plt.subplots(figsize=(10, 8), facecolor='white')

cmap = plt.get_cmap('RdBu_r')   # colorblind-friendly diverging

for row_i, (feat_name, feat_idx) in enumerate(zip(top15_rev, top15_idx_rev)):
    sv   = shap_s[:, feat_idx]
    fv   = X_s[:, feat_idx]

    # Normalize feature values to [0,1] for coloring
    fv_norm = (fv - fv.min()) / (fv.max() - fv.min() + 1e-9)

    # Jitter y for beeswarm effect
    jitter = rng.uniform(-0.35, 0.35, size=len(sv))
    colors = cmap(fv_norm)

    ax.scatter(sv, row_i + jitter, c=colors, s=6, alpha=0.55,
               linewidths=0, rasterized=True)

    # Mean |SHAP| bar (background)
    mean_sv = np.abs(sv).mean()
    ax.barh(row_i, mean_sv, height=0.55, color='#dddddd',
            alpha=0.4, zorder=0, left=0)

ax.axvline(0, color='#333333', lw=1.0, linestyle='--', alpha=0.7)
ax.set_yticks(range(len(top15_rev)))
ax.set_yticklabels([nice_labels.get(f, f) for f in top15_rev], fontsize=10)
ax.set_xlabel('SHAP Value (impact on model output)', fontsize=11)
ax.set_title('Figure 4 — SHAP Beeswarm: Top 15 Features\n(SIMULATED DATA, N = 19,200 | XGBoost)',
             fontsize=12, fontweight='bold')
ax.spines[['top','right']].set_visible(False)
ax.tick_params(axis='x', labelsize=9)

# Colorbar for feature value
sm = cm.ScalarMappable(cmap=cmap, norm=Normalize(vmin=0, vmax=1))
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax, orientation='vertical', fraction=0.025, pad=0.02)
cbar.set_label('Feature Value\n(low → high)', fontsize=9)
cbar.set_ticks([0, 0.5, 1])
cbar.set_ticklabels(['Low', 'Mid', 'High'], fontsize=8)

fig.text(0.99, 0.01, 'SIMULATED DATA — for methodology demonstration only',
         ha='right', va='bottom', fontsize=7, color='#888888', style='italic')

plt.tight_layout()
plt.savefig('/mnt/results/study1/figures/Figure4.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.savefig('/mnt/results/study1/figures/Figure4.svg', bbox_inches='tight', facecolor='white')
plt.close()
print("Figure4 saved.")


Figure4 saved.


Study 1 Figure 5 — Confusion matrix heatmap (4-class + binary)

In [17]:

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold, cross_val_predict
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

df   = pd.read_csv('/mnt/results/study1/data/LNPDB_SIMULATED_N19200.csv')
cv_r = pd.read_csv('/mnt/results/study1/data/LNPDB_SIMULATED_cv_predictions.csv')

feat_cols = ['CHL_mol_pct','HL_mol_pct','PEG_mol_pct','IL_mol_pct',
             'size_nm','PDI','zeta_mV','encap_eff_pct','N_P_ratio',
             'Frac_sp3C','MW_Da','logP','HBD','HBA','pKa','TPSA',
             'RotBonds','AromaticRings']
X = df[feat_cols].values

# --- Binary confusion matrix ---
y_bin_true = cv_r['y_true'].values
y_bin_pred = cv_r['y_pred'].values
cm_bin = confusion_matrix(y_bin_true, y_bin_pred)
cm_bin_norm = cm_bin.astype(float) / cm_bin.sum(axis=1, keepdims=True)

# --- 4-class confusion matrix ---
classes = ['Very Low','Low','High','Very High']
le = LabelEncoder(); le.fit(classes)
y_enc = le.transform(df['efficacy_class'].values)

model_mc = xgb.XGBClassifier(
    n_estimators=300, max_depth=5, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    min_child_weight=5, gamma=1.0,
    reg_alpha=0.5, reg_lambda=2.0,
    objective='multi:softprob', num_class=4,
    eval_metric='mlogloss', random_state=42, n_jobs=-1
)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
y_pred_mc = cross_val_predict(model_mc, X, y_enc, cv=cv, method='predict')
cm_mc = confusion_matrix(y_enc, y_pred_mc)
cm_mc_norm = cm_mc.astype(float) / cm_mc.sum(axis=1, keepdims=True)

# --- Plot ---
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5), facecolor='white')

# Panel A: Binary
ax = axes[0]
annot_bin = np.array([[f'{cm_bin[i,j]}\n({cm_bin_norm[i,j]:.1%})'
                        for j in range(2)] for i in range(2)])
sns.heatmap(cm_bin_norm, ax=ax, annot=annot_bin, fmt='',
            cmap='Blues', vmin=0, vmax=1,
            xticklabels=['Low\n(pred)', 'High\n(pred)'],
            yticklabels=['Low\n(true)', 'High\n(true)'],
            linewidths=1, linecolor='white',
            cbar_kws={'label': 'Recall (row-normalized)', 'shrink': 0.8},
            annot_kws={'size': 12, 'weight': 'bold'})
ax.set_title('A  Binary Confusion Matrix\n(High vs Low Efficacy)',
             fontsize=11, fontweight='bold', loc='left')
ax.set_xlabel('Predicted Label', fontsize=10)
ax.set_ylabel('True Label', fontsize=10)
ax.tick_params(labelsize=9)

# Panel B: 4-class
ax2 = axes[1]
annot_mc = np.array([[f'{cm_mc[i,j]}\n({cm_mc_norm[i,j]:.1%})'
                       for j in range(4)] for i in range(4)])
sns.heatmap(cm_mc_norm, ax=ax2, annot=annot_mc, fmt='',
            cmap='Blues', vmin=0, vmax=1,
            xticklabels=[f'{c}\n(pred)' for c in classes],
            yticklabels=[f'{c}\n(true)' for c in classes],
            linewidths=1, linecolor='white',
            cbar_kws={'label': 'Recall (row-normalized)', 'shrink': 0.8},
            annot_kws={'size': 9})
ax2.set_title('B  4-Class Confusion Matrix\n(Efficacy Classes)',
              fontsize=11, fontweight='bold', loc='left')
ax2.set_xlabel('Predicted Label', fontsize=10)
ax2.set_ylabel('True Label', fontsize=10)
ax2.tick_params(axis='x', labelsize=8, rotation=15)
ax2.tick_params(axis='y', labelsize=8, rotation=0)

fig.suptitle('Figure 5 — Confusion Matrices: XGBoost Transfection Efficacy Classifier\n(SIMULATED DATA, N = 19,200 | 5-fold CV)',
             fontsize=12, fontweight='bold', y=1.02)

fig.text(0.99, 0.01, 'SIMULATED DATA — for methodology demonstration only',
         ha='right', va='bottom', fontsize=7, color='#888888', style='italic')

plt.tight_layout()
plt.savefig('/mnt/results/study1/figures/Figure5.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.savefig('/mnt/results/study1/figures/Figure5.svg', bbox_inches='tight', facecolor='white')
plt.close()
print("Figure5 saved.")


Figure5 saved.


Study 1 FigureS1 — Corona proof-of-concept model (N=26, LOOCV AUC=0.794)

In [19]:

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc, confusion_matrix
from sklearn.model_selection import LeaveOneOut
from sklearn.ensemble import RandomForestClassifier
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

np.random.seed(99)
N_corona = 26

# Simulated corona composition dataset (N=26 formulations)
# Features: CHL mol%, HL mol%, PEG mol%, size, PDI, zeta
# Target: high_uptake (binary) — driven by ApoE enrichment
CHL  = np.random.normal(40, 5, N_corona).clip(28, 52)
HL   = np.random.normal(10, 2.5, N_corona).clip(5, 18)
PEG  = np.random.normal(2.5, 0.6, N_corona).clip(0.8, 4.5)
size = np.random.lognormal(np.log(95), 0.25, N_corona).clip(55, 180)
PDI  = np.random.beta(2, 9, N_corona).clip(0.06, 0.35)
zeta = np.random.normal(-4, 6, N_corona).clip(-25, 15)

# ApoE enrichment (% corona) — key predictor
ApoE_pct    = np.random.normal(18, 6, N_corona).clip(5, 40)
albumin_pct = np.random.normal(45, 10, N_corona).clip(20, 70)
vitronectin = np.random.normal(8, 3, N_corona).clip(2, 18)
fibronectin = np.random.normal(6, 2.5, N_corona).clip(1, 14)

# Label: high uptake driven by ApoE > 18% and small PDI
score = (ApoE_pct - 18) / 6 - PDI / 0.15 + np.random.normal(0, 0.4, N_corona)
threshold = np.percentile(score, 50)
y_corona = (score >= threshold).astype(int)

X_corona = np.column_stack([CHL, HL, PEG, size, PDI, zeta,
                              ApoE_pct, albumin_pct, vitronectin, fibronectin])
feat_names = ['CHL mol%','HL mol%','PEG mol%','Size (nm)','PDI','Zeta (mV)',
              'ApoE %','Albumin %','Vitronectin %','Fibronectin %']

# LOOCV with Random Forest (small N → RF more stable than XGB)
loo = LeaveOneOut()
rf = RandomForestClassifier(n_estimators=200, max_depth=3, random_state=42)

y_prob_loo = np.zeros(N_corona)
y_pred_loo = np.zeros(N_corona, dtype=int)
for train_idx, test_idx in loo.split(X_corona):
    rf.fit(X_corona[train_idx], y_corona[train_idx])
    y_prob_loo[test_idx] = rf.predict_proba(X_corona[test_idx])[:, 1]
    y_pred_loo[test_idx] = rf.predict(X_corona[test_idx])

fpr, tpr, _ = roc_curve(y_corona, y_prob_loo)
roc_auc = auc(fpr, tpr)
cm = confusion_matrix(y_corona, y_pred_loo)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

# Feature importances (fit on full data)
rf.fit(X_corona, y_corona)
importances = rf.feature_importances_
imp_df = pd.DataFrame({'feature': feat_names, 'importance': importances})
imp_df = imp_df.sort_values('importance', ascending=True)

# Save corona dataset
corona_df = pd.DataFrame({
    'formulation_id': [f'CORONA_{i:02d}' for i in range(N_corona)],
    'CHL_mol_pct': np.round(CHL, 2), 'HL_mol_pct': np.round(HL, 2),
    'PEG_mol_pct': np.round(PEG, 2), 'size_nm': np.round(size, 1),
    'PDI': np.round(PDI, 3), 'zeta_mV': np.round(zeta, 1),
    'ApoE_pct': np.round(ApoE_pct, 1), 'albumin_pct': np.round(albumin_pct, 1),
    'vitronectin_pct': np.round(vitronectin, 1), 'fibronectin_pct': np.round(fibronectin, 1),
    'high_uptake': y_corona, 'loocv_prob': np.round(y_prob_loo, 4)
})
corona_df.to_csv('/mnt/results/study1/data/LNPDB_SIMULATED_corona_N26.csv', index=False)

# --- Plot ---
CB = ['#e41a1c','#377eb8','#4daf4a','#984ea3']
fig, axes = plt.subplots(1, 3, figsize=(15, 5.5), facecolor='white')

# Panel A: ROC
ax = axes[0]
ax.plot(fpr, tpr, color=CB[1], lw=2.5,
        label=f'LOOCV AUC = {roc_auc:.3f}')
ax.fill_between(fpr, tpr, alpha=0.12, color=CB[1])
ax.plot([0,1],[0,1], 'k--', lw=1, alpha=0.5, label='Random (0.500)')
ax.set_xlabel('False Positive Rate', fontsize=11)
ax.set_ylabel('True Positive Rate', fontsize=11)
ax.set_title('A  LOOCV ROC Curve\n(N = 26 formulations)',
             fontsize=11, fontweight='bold', loc='left')
ax.legend(fontsize=10, loc='lower right')
ax.set_xlim([0,1]); ax.set_ylim([0,1.02])
ax.spines[['top','right']].set_visible(False)
ax.tick_params(labelsize=9)
ax.text(0.55, 0.15, f'AUC = {roc_auc:.3f}\n(proof-of-concept)',
        fontsize=10, color=CB[1], fontweight='bold',
        transform=ax.transAxes)

# Panel B: Confusion matrix
ax2 = axes[1]
annot = np.array([[f'{cm[i,j]}\n({cm_norm[i,j]:.0%})'
                   for j in range(2)] for i in range(2)])
sns.heatmap(cm_norm, ax=ax2, annot=annot, fmt='',
            cmap='Blues', vmin=0, vmax=1,
            xticklabels=['Low\n(pred)','High\n(pred)'],
            yticklabels=['Low\n(true)','High\n(true)'],
            linewidths=1.5, linecolor='white',
            cbar_kws={'label': 'Recall', 'shrink': 0.8},
            annot_kws={'size': 13, 'weight': 'bold'})
ax2.set_title('B  Confusion Matrix\n(LOOCV)',
              fontsize=11, fontweight='bold', loc='left')
ax2.set_xlabel('Predicted', fontsize=10)
ax2.set_ylabel('True', fontsize=10)
ax2.tick_params(labelsize=9)

# Panel C: Feature importances
ax3 = axes[2]
colors_imp = [CB[0] if imp > imp_df['importance'].median() else CB[2]
              for imp in imp_df['importance']]
ax3.barh(imp_df['feature'], imp_df['importance'],
         color=colors_imp, edgecolor='white', linewidth=0.5)
ax3.set_xlabel('Feature Importance (Gini)', fontsize=10)
ax3.set_title('C  Feature Importances\n(Random Forest, full data)',
              fontsize=11, fontweight='bold', loc='left')
ax3.spines[['top','right']].set_visible(False)
ax3.tick_params(labelsize=9)

fig.suptitle('Figure S1 — Corona Composition Proof-of-Concept Model\n'
             '(SIMULATED DATA, N = 26 | LOOCV | Random Forest)',
             fontsize=12, fontweight='bold', y=1.02)

fig.text(0.99, 0.01, 'SIMULATED DATA — proof-of-concept only, N insufficient for publication',
         ha='right', va='bottom', fontsize=7, color='#cc0000', style='italic', fontweight='bold')

plt.tight_layout()
plt.savefig('/mnt/results/study1/figures/FigureS1.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.savefig('/mnt/results/study1/figures/FigureS1.svg', bbox_inches='tight', facecolor='white')
plt.close()
print(f"FigureS1 saved. LOOCV AUC = {roc_auc:.3f} (target: ~0.794)")


FigureS1 saved. LOOCV AUC = 0.834 (target: ~0.794)


Generate Study 2 matched static/dynamic corona dataset (N=32), LNPDB stability ranking (N=100), Vroman kinetics

In [21]:

import numpy as np
import pandas as pd

np.random.seed(2026)
N = 32

# --- Formulation types ---
lipid_types = ['Ionizable-cationic', 'Ionizable-neutral', 'Helper-only', 'PEGylated']
lipid_charge = np.random.choice(['cationic','neutral','anionic'], N, p=[0.45, 0.35, 0.20])
formulation_type = np.random.choice(['MC3-LNP','SM102-LNP','ALC0315-LNP',
                                      'C12200-LNP','DOTAP-LNP','Lipid5-LNP',
                                      'cKKE12-LNP','OF02-LNP'], N)
cargo = np.random.choice(['mRNA','siRNA','pDNA'], N, p=[0.50, 0.35, 0.15])

# --- Static corona measurements ---
CHL_mol   = np.random.normal(40, 5, N).clip(28, 52)
HL_mol    = np.random.normal(10, 2.5, N).clip(5, 18)
PEG_mol   = np.random.normal(2.5, 0.6, N).clip(0.8, 4.5)
size_nm   = np.random.lognormal(np.log(95), 0.22, N).clip(55, 180)
PDI_stat  = np.random.beta(2, 9, N).clip(0.06, 0.32)
zeta_stat = np.random.normal(-4, 7, N).clip(-28, 18)

albumin_static = np.random.normal(48, 9, N).clip(25, 70)
ApoE_static    = np.random.normal(14, 5, N).clip(3, 32)
fibronectin_s  = np.random.normal(7, 2.5, N).clip(1, 15)
vitronectin_s  = np.random.normal(6, 2.0, N).clip(1, 13)
IgG_s          = np.random.normal(8, 3, N).clip(1, 18)

# --- Dynamic (flow) corona — literature-anchored deltas ---
# delta_zeta = -5.4 ± 4.0 mV (flow → more negative)
# delta_albumin = -12.2 ± 9.3 pp (albumin displaced)
# delta_ApoE = +6.0 ± 4.3 pp (ApoE enriched)
delta_zeta    = np.random.normal(-5.4, 4.0, N).clip(-18, 2)
delta_albumin = np.random.normal(-12.2, 9.3, N).clip(-35, 5)
delta_ApoE    = np.random.normal(6.0, 4.3, N).clip(-3, 20)
delta_PDI     = np.random.normal(0.02, 0.015, N).clip(-0.02, 0.08)

zeta_flow    = (zeta_stat + delta_zeta).clip(-35, 15)
albumin_flow = (albumin_static + delta_albumin).clip(10, 70)
ApoE_flow    = (ApoE_static + delta_ApoE).clip(2, 45)
PDI_flow     = (PDI_stat + delta_PDI).clip(0.05, 0.45)

# Corona shift index: composite of |delta_zeta| + |delta_albumin| + delta_ApoE
corona_shift_index = (
    np.abs(delta_zeta) / 5.4 * 0.25 +
    np.abs(delta_albumin) / 12.2 * 0.45 +
    delta_ApoE / 6.0 * 0.30
)
corona_shift_index = np.round(corona_shift_index, 3)

df2 = pd.DataFrame({
    'formulation_id':    [f'DYN_{i:02d}' for i in range(N)],
    'formulation_type':  formulation_type,
    'lipid_charge':      lipid_charge,
    'cargo':             cargo,
    'CHL_mol_pct':       np.round(CHL_mol, 2),
    'HL_mol_pct':        np.round(HL_mol, 2),
    'PEG_mol_pct':       np.round(PEG_mol, 2),
    'size_nm':           np.round(size_nm, 1),
    # Static
    'PDI_static':        np.round(PDI_stat, 3),
    'zeta_static_mV':    np.round(zeta_stat, 1),
    'albumin_static_pp': np.round(albumin_static, 1),
    'ApoE_static_pp':    np.round(ApoE_static, 1),
    'fibronectin_s_pp':  np.round(fibronectin_s, 1),
    'vitronectin_s_pp':  np.round(vitronectin_s, 1),
    'IgG_static_pp':     np.round(IgG_s, 1),
    # Dynamic
    'PDI_flow':          np.round(PDI_flow, 3),
    'zeta_flow_mV':      np.round(zeta_flow, 1),
    'albumin_flow_pp':   np.round(albumin_flow, 1),
    'ApoE_flow_pp':      np.round(ApoE_flow, 1),
    # Deltas
    'delta_zeta_mV':     np.round(delta_zeta, 2),
    'delta_PDI':         np.round(delta_PDI, 3),
    'delta_albumin_pp':  np.round(delta_albumin, 1),
    'delta_ApoE_pp':     np.round(delta_ApoE, 1),
    'corona_shift_index': corona_shift_index,
})

df2.to_csv('/mnt/results/study2/data/DYNAMIC_CORONA_SIMULATED_N32.csv', index=False)

print(f"Dataset shape: {df2.shape}")
print(f"\nDelta statistics (should match spec):")
print(f"  delta_zeta:    {df2['delta_zeta_mV'].mean():.2f} ± {df2['delta_zeta_mV'].std():.2f} mV  (spec: -5.4 ± 4.0)")
print(f"  delta_albumin: {df2['delta_albumin_pp'].mean():.2f} ± {df2['delta_albumin_pp'].std():.2f} pp  (spec: -12.2 ± 9.3)")
print(f"  delta_ApoE:    {df2['delta_ApoE_pp'].mean():.2f} ± {df2['delta_ApoE_pp'].std():.2f} pp  (spec: +6.0 ± 4.3)")
print(f"\ncorona_shift_index: {df2['corona_shift_index'].describe().round(3)}")

# --- LNPDB stability ranking (N=100) ---
np.random.seed(777)
N_rank = 100
stab_df = pd.DataFrame({
    'formulation_id':    [f'LNPDB_{i:03d}' for i in range(N_rank)],
    'formulation_type':  np.random.choice(['MC3-LNP','SM102-LNP','ALC0315-LNP',
                                            'C12200-LNP','DOTAP-LNP','Lipid5-LNP',
                                            'cKKE12-LNP','OF02-LNP'], N_rank),
    'CHL_mol_pct':       np.round(np.random.normal(40, 6, N_rank).clip(25, 55), 2),
    'PEG_mol_pct':       np.round(np.random.normal(2.5, 0.8, N_rank).clip(0.5, 5), 2),
    'corona_shift_index': np.round(np.abs(np.random.normal(1.0, 0.5, N_rank)).clip(0.1, 2.8), 3),
})
stab_df = stab_df.sort_values('corona_shift_index').reset_index(drop=True)
stab_df.to_csv('/mnt/results/study2/data/LNPDB_SIMULATED_stability_ranking_N100.csv', index=False)
print(f"\nStability ranking saved: {stab_df.shape}")

# --- Vroman kinetics time-course ---
t = np.arange(0, 61, 2)   # 0–60 min, every 2 min
k_fast_static  = 0.03   # min⁻¹
k_fast_flow    = 0.10   # min⁻¹  (3-4x faster)
k_slow_static  = 0.005
k_slow_flow    = 0.015

# Albumin: starts high, decays (displaced by ApoE)
albumin_eq_static = 48.0
albumin_eq_flow   = 36.0
albumin_s_t = albumin_eq_static + (60 - albumin_eq_static) * np.exp(-k_fast_static * t)
albumin_f_t = albumin_eq_flow   + (60 - albumin_eq_flow)   * np.exp(-k_fast_flow   * t)

# ApoE: starts low, rises (Vroman exchange)
ApoE_eq_static = 14.0
ApoE_eq_flow   = 20.0
ApoE_s_t = ApoE_eq_static * (1 - np.exp(-k_slow_static * t))
ApoE_f_t = ApoE_eq_flow   * (1 - np.exp(-k_slow_flow   * t))

# Add small noise
rng = np.random.default_rng(42)
noise = lambda s: rng.normal(0, s, len(t))
albumin_s_t += noise(1.2); albumin_f_t += noise(1.5)
ApoE_s_t    += noise(0.6); ApoE_f_t    += noise(0.8)

vroman_df = pd.DataFrame({
    'time_min':          t,
    'albumin_static_pp': np.round(albumin_s_t.clip(20, 70), 2),
    'albumin_flow_pp':   np.round(albumin_f_t.clip(15, 65), 2),
    'ApoE_static_pp':    np.round(ApoE_s_t.clip(0, 35), 2),
    'ApoE_flow_pp':      np.round(ApoE_f_t.clip(0, 40), 2),
})
vroman_df.to_csv('/mnt/results/study2/data/VROMAN_SIMULATED_kinetics.csv', index=False)
print(f"\nVroman kinetics saved: {vroman_df.shape}")
print(vroman_df.tail(5))


Dataset shape: (32, 24)

Delta statistics (should match spec):
  delta_zeta:    -4.90 ± 4.41 mV  (spec: -5.4 ± 4.0)
  delta_albumin: -16.87 ± 9.43 pp  (spec: -12.2 ± 9.3)
  delta_ApoE:    7.14 ± 4.64 pp  (spec: +6.0 ± 4.3)

corona_shift_index: count    32.000
mean      1.237
std       0.462
min       0.206
25%       0.956
50%       1.248
75%       1.581
max       2.210
Name: corona_shift_index, dtype: float64

Stability ranking saved: (100, 5)

Vroman kinetics saved: (31, 5)
    time_min  albumin_static_pp  albumin_flow_pp  ApoE_static_pp  ApoE_flow_pp
26        52              51.16            38.37            3.49         10.64
27        54              50.81            34.81            3.58         10.28
28        56              50.73            37.54            3.82         11.51
29        58              50.62            33.55            3.47         11.80
30        60              52.55            35.56            3.37         12.96


Study 2 Figure 1 — Conceptual schematic: static lab vs in vivo flow (text-based diagram)

In [23]:

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyArrowPatch, FancyBboxPatch, Circle, Ellipse
import matplotlib.patheffects as pe
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(14, 8), facecolor='white')
fig.suptitle('Figure 1 — Static Lab Incubation vs. Physiological Blood Flow\n'
             'Protein Corona Formation in LNPs',
             fontsize=13, fontweight='bold', y=0.98)

CB = ['#e41a1c','#377eb8','#4daf4a','#984ea3','#ff7f00','#a65628','#f781bf','#999999']

# ─────────────────────────────────────────────
# LEFT PANEL: Static incubation tube
# ─────────────────────────────────────────────
ax = axes[0]
ax.set_xlim(0, 10); ax.set_ylim(0, 12)
ax.set_aspect('equal')
ax.axis('off')
ax.set_facecolor('white')
ax.set_title('A   Static Lab Incubation', fontsize=12, fontweight='bold',
             color='#377eb8', pad=10)

# Tube outline
tube = FancyBboxPatch((3.5, 1.0), 3.0, 8.5,
                       boxstyle="round,pad=0.15",
                       linewidth=2, edgecolor='#555555',
                       facecolor='#f0f8ff', zorder=1)
ax.add_patch(tube)

# Serum fill
serum = FancyBboxPatch((3.6, 1.1), 2.8, 6.5,
                        boxstyle="round,pad=0.05",
                        linewidth=0, facecolor='#ffe8cc', alpha=0.7, zorder=2)
ax.add_patch(serum)
ax.text(5.0, 4.5, 'Serum\n37°C', ha='center', va='center',
        fontsize=9, color='#cc6600', style='italic', zorder=5)

# LNP particles (circles)
rng = np.random.default_rng(10)
for _ in range(12):
    cx = rng.uniform(4.0, 6.5)
    cy = rng.uniform(1.5, 7.0)
    c = Circle((cx, cy), 0.22, color='#377eb8', zorder=6, alpha=0.85)
    ax.add_patch(c)
    # Corona ring
    cr = Circle((cx, cy), 0.32, color='#e41a1c', fill=False,
                linewidth=1.5, zorder=5, alpha=0.7)
    ax.add_patch(cr)

# Albumin label
ax.annotate('Albumin-rich\ncorona', xy=(6.3, 5.5), xytext=(7.5, 6.5),
            fontsize=8, color='#e41a1c',
            arrowprops=dict(arrowstyle='->', color='#e41a1c', lw=1.2))

# Tube cap
cap = FancyBboxPatch((3.4, 9.3), 3.2, 0.5,
                      boxstyle="round,pad=0.1",
                      linewidth=1.5, edgecolor='#555555', facecolor='#cccccc', zorder=7)
ax.add_patch(cap)

# Conditions box
cond = FancyBboxPatch((0.3, 0.2), 9.4, 1.2,
                       boxstyle="round,pad=0.1",
                       linewidth=1, edgecolor='#377eb8', facecolor='#e8f4fd', zorder=3)
ax.add_patch(cond)
ax.text(5.0, 0.82, 'Shear stress: 0 dyn/cm²  |  Incubation: 1–4 h  |  Static',
        ha='center', va='center', fontsize=8, color='#1a5276', fontweight='bold')

# No-flow symbol
ax.text(5.0, 10.5, '⊘ No Flow', ha='center', va='center',
        fontsize=13, color='#e41a1c', fontweight='bold')

# ─────────────────────────────────────────────
# RIGHT PANEL: Blood vessel with flow
# ─────────────────────────────────────────────
ax2 = axes[1]
ax2.set_xlim(0, 10); ax2.set_ylim(0, 12)
ax2.set_aspect('equal')
ax2.axis('off')
ax2.set_facecolor('white')
ax2.set_title('B   Physiological Blood Flow (In Vivo)', fontsize=12, fontweight='bold',
              color='#e41a1c', pad=10)

# Vessel walls
vessel_top = FancyBboxPatch((0.5, 7.5), 9.0, 1.2,
                             boxstyle="round,pad=0.05",
                             linewidth=2, edgecolor='#8B0000', facecolor='#ffcccc', zorder=1)
vessel_bot = FancyBboxPatch((0.5, 3.3), 9.0, 1.2,
                             boxstyle="round,pad=0.05",
                             linewidth=2, edgecolor='#8B0000', facecolor='#ffcccc', zorder=1)
ax2.add_patch(vessel_top)
ax2.add_patch(vessel_bot)

# Blood lumen
lumen = FancyBboxPatch((0.5, 4.5), 9.0, 3.0,
                        boxstyle="round,pad=0.0",
                        linewidth=0, facecolor='#ffe0e0', alpha=0.6, zorder=0)
ax2.add_patch(lumen)

# Flow arrows (parabolic profile)
arrow_ys = [5.0, 5.5, 6.0, 6.5, 7.0]
arrow_lens = [0.8, 1.4, 1.8, 1.4, 0.8]
for ay, al in zip(arrow_ys, arrow_lens):
    ax2.annotate('', xy=(2.5 + al*2.5, ay), xytext=(2.0, ay),
                 arrowprops=dict(arrowstyle='->', color='#cc0000',
                                 lw=1.5 + al*0.3))

ax2.text(5.5, 6.0, 'Blood Flow', ha='center', va='center',
         fontsize=10, color='#8B0000', fontweight='bold', style='italic', zorder=10)

# LNP particles with ApoE-enriched corona
rng2 = np.random.default_rng(20)
for _ in range(8):
    cx = rng2.uniform(1.5, 8.5)
    cy = rng2.uniform(4.7, 7.3)
    c = Circle((cx, cy), 0.22, color='#377eb8', zorder=8, alpha=0.9)
    ax2.add_patch(c)
    cr = Circle((cx, cy), 0.33, color='#4daf4a', fill=False,
                linewidth=2.0, zorder=7, alpha=0.85)
    ax2.add_patch(cr)

ax2.annotate('ApoE-enriched\ncorona', xy=(7.2, 5.8), xytext=(7.8, 4.0),
             fontsize=8, color='#4daf4a', fontweight='bold',
             arrowprops=dict(arrowstyle='->', color='#4daf4a', lw=1.2))

# Shear stress values
ax2.text(5.0, 8.9, 'Venous: 1–4 dyn/cm²  |  Arterial: 10–70 dyn/cm²',
         ha='center', va='center', fontsize=8, color='#8B0000',
         bbox=dict(boxstyle='round,pad=0.3', facecolor='#fff0f0', edgecolor='#cc0000', lw=1))

# Conditions box
cond2 = FancyBboxPatch((0.3, 0.2), 9.4, 1.2,
                        boxstyle="round,pad=0.1",
                        linewidth=1, edgecolor='#e41a1c', facecolor='#fde8e8', zorder=3)
ax2.add_patch(cond2)
ax2.text(5.0, 0.82, 'Shear stress: 1–70 dyn/cm²  |  Continuous flow  |  Dynamic exchange',
         ha='center', va='center', fontsize=8, color='#8B0000', fontweight='bold')

# "Gap this study addresses" text box
gap_box = FancyBboxPatch((0.8, 9.5), 8.4, 1.8,
                          boxstyle="round,pad=0.2",
                          linewidth=2, edgecolor='#984ea3', facecolor='#f5eef8', zorder=10)
ax2.add_patch(gap_box)
ax2.text(5.0, 10.4, '⚡ Gap this study addresses:',
         ha='center', va='center', fontsize=9, color='#6c3483', fontweight='bold')
ax2.text(5.0, 9.85,
         'Static assays underestimate ApoE enrichment by 3–4×\n'
         '→ Systematic error in LNP brain-targeting prediction',
         ha='center', va='center', fontsize=8, color='#4a235a', style='italic')

# Legend
legend_elements = [
    mpatches.Patch(facecolor='#377eb8', label='LNP core'),
    mpatches.Patch(facecolor='#e41a1c', alpha=0.5, label='Albumin corona (static)'),
    mpatches.Patch(facecolor='#4daf4a', alpha=0.7, label='ApoE corona (flow)'),
]
fig.legend(handles=legend_elements, loc='lower center', ncol=3,
           fontsize=9, frameon=True, bbox_to_anchor=(0.5, 0.01),
           edgecolor='#cccccc')

# SIMULATED watermark
fig.text(0.99, 0.04, 'SIMULATED DATA',
         ha='right', va='bottom', fontsize=11, color='#cc0000',
         fontweight='bold', alpha=0.55, rotation=0,
         bbox=dict(boxstyle='round,pad=0.3', facecolor='#fff0f0',
                   edgecolor='#cc0000', lw=1.5, alpha=0.7))

plt.tight_layout(rect=[0, 0.06, 1, 0.97])
plt.savefig('/mnt/results/study2/figures/Figure1.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.savefig('/mnt/results/study2/figures/Figure1.svg', bbox_inches='tight', facecolor='white')
plt.close()
print("Study2 Figure1 saved.")


Study2 Figure1 saved.


Study 2 Figures 2 & 3 — Dataset overview grouped bar + delta distributions violin plots

In [25]:

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

df2 = pd.read_csv('/mnt/results/study2/data/DYNAMIC_CORONA_SIMULATED_N32.csv')

CB = ['#e41a1c','#377eb8','#4daf4a','#984ea3','#ff7f00','#a65628','#f781bf','#999999']

WATERMARK_KWARGS = dict(ha='right', va='bottom', fontsize=10, color='#cc0000',
                        fontweight='bold', alpha=0.6,
                        bbox=dict(boxstyle='round,pad=0.3', facecolor='#fff0f0',
                                  edgecolor='#cc0000', lw=1.5, alpha=0.7))

# ══════════════════════════════════════════════
# FIGURE 2 — Dataset overview: grouped bar chart
# ══════════════════════════════════════════════
fig2, axes2 = plt.subplots(1, 2, figsize=(13, 6), facecolor='white')
fig2.suptitle('Figure 2 — Study Dataset Overview: Matched Static/Dynamic Pairs\n(SIMULATED DATA, N = 32 formulations)',
              fontsize=12, fontweight='bold', y=1.01)

# Panel A: N per formulation type
ax = axes2[0]
ftype_counts = df2['formulation_type'].value_counts().sort_values(ascending=False)
bars = ax.bar(range(len(ftype_counts)), ftype_counts.values,
              color=CB[:len(ftype_counts)], edgecolor='white', linewidth=0.5)
ax.set_xticks(range(len(ftype_counts)))
ax.set_xticklabels(ftype_counts.index, rotation=35, ha='right', fontsize=8)
ax.set_ylabel('Number of Formulations', fontsize=10)
ax.set_title('A  Formulations per LNP Type', fontsize=11, fontweight='bold', loc='left')
ax.spines[['top','right']].set_visible(False)
for bar, val in zip(bars, ftype_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.05, str(val),
            ha='center', va='bottom', fontsize=9, fontweight='bold')
ax.set_ylim(0, ftype_counts.max() * 1.25)

# Panel B: N per lipid charge type × cargo
ax2b = axes2[1]
charge_cargo = df2.groupby(['lipid_charge','cargo']).size().unstack(fill_value=0)
x = np.arange(len(charge_cargo.index))
width = 0.25
for i, (col, color) in enumerate(zip(charge_cargo.columns, CB)):
    offset = (i - len(charge_cargo.columns)/2 + 0.5) * width
    bars2 = ax2b.bar(x + offset, charge_cargo[col], width,
                     label=col, color=color, edgecolor='white', linewidth=0.5)
ax2b.set_xticks(x)
ax2b.set_xticklabels(charge_cargo.index, fontsize=10)
ax2b.set_ylabel('Number of Formulations', fontsize=10)
ax2b.set_title('B  Formulations by Lipid Charge × Cargo', fontsize=11, fontweight='bold', loc='left')
ax2b.legend(title='Cargo', fontsize=9, title_fontsize=9, frameon=True)
ax2b.spines[['top','right']].set_visible(False)
ax2b.tick_params(labelsize=9)

fig2.text(0.99, 0.01, 'SIMULATED DATA', **WATERMARK_KWARGS)
plt.tight_layout()
plt.savefig('/mnt/results/study2/figures/Figure2.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.savefig('/mnt/results/study2/figures/Figure2.svg', bbox_inches='tight', facecolor='white')
plt.close()
print("Figure2 saved.")

# ══════════════════════════════════════════════
# FIGURE 3 — Delta distributions: 4-panel violin
# ══════════════════════════════════════════════
fig3, axes3 = plt.subplots(2, 2, figsize=(12, 9), facecolor='white')
fig3.suptitle('Figure 3 — Corona Shift Distributions: Static → Flow\n(SIMULATED DATA, N = 32 | Colored by Lipid Charge Type)',
              fontsize=12, fontweight='bold', y=1.01)

charge_palette = {'cationic': CB[0], 'neutral': CB[1], 'anionic': CB[2]}

panels = [
    ('delta_zeta_mV',    'Δ Zeta Potential (mV)',       'A  Δ Zeta (mV)',       axes3[0,0]),
    ('delta_PDI',        'Δ PDI',                        'B  Δ PDI',             axes3[0,1]),
    ('delta_ApoE_pp',    'Δ ApoE (percentage points)',   'C  Δ ApoE (%)',        axes3[1,0]),
    ('delta_albumin_pp', 'Δ Albumin (percentage points)','D  Δ Albumin (%)',     axes3[1,1]),
]

for col, ylabel, title, ax in panels:
    # Violin
    charges = df2['lipid_charge'].unique()
    positions = np.arange(len(charges))
    data_by_charge = [df2[df2['lipid_charge'] == c][col].values for c in charges]

    parts = ax.violinplot(data_by_charge, positions=positions,
                          showmedians=True, showextrema=True)
    for i, (pc, charge) in enumerate(zip(parts['bodies'], charges)):
        pc.set_facecolor(charge_palette[charge])
        pc.set_alpha(0.65)
    parts['cmedians'].set_color('#333333')
    parts['cmedians'].set_linewidth(2)
    parts['cbars'].set_color('#555555')
    parts['cmaxes'].set_color('#555555')
    parts['cmins'].set_color('#555555')

    # Jittered points
    for i, (data, charge) in enumerate(zip(data_by_charge, charges)):
        jitter = np.random.default_rng(i).uniform(-0.12, 0.12, len(data))
        ax.scatter(i + jitter, data, color=charge_palette[charge],
                   s=25, alpha=0.8, zorder=5, edgecolors='white', linewidths=0.5)

    ax.axhline(0, color='#333333', lw=1.2, linestyle='--', alpha=0.6)
    ax.set_xticks(positions)
    ax.set_xticklabels(charges, fontsize=10)
    ax.set_ylabel(ylabel, fontsize=9)
    ax.set_title(title, fontsize=11, fontweight='bold', loc='left')
    ax.spines[['top','right']].set_visible(False)
    ax.tick_params(labelsize=9)

    # Annotate mean ± SD
    overall_mean = df2[col].mean()
    overall_std  = df2[col].std()
    ax.text(0.97, 0.05, f'Mean: {overall_mean:+.1f} ± {overall_std:.1f}',
            transform=ax.transAxes, ha='right', fontsize=8,
            color='#333333', style='italic',
            bbox=dict(boxstyle='round,pad=0.2', facecolor='#f9f9f9',
                      edgecolor='#cccccc', lw=0.8))

legend_patches = [mpatches.Patch(color=v, label=k) for k, v in charge_palette.items()]
fig3.legend(handles=legend_patches, title='Lipid Charge', loc='lower center',
            ncol=3, fontsize=10, title_fontsize=10, frameon=True,
            bbox_to_anchor=(0.5, 0.0))

fig3.text(0.99, 0.02, 'SIMULATED DATA', **WATERMARK_KWARGS)
plt.tight_layout(rect=[0, 0.05, 1, 1])
plt.savefig('/mnt/results/study2/figures/Figure3.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.savefig('/mnt/results/study2/figures/Figure3.svg', bbox_inches='tight', facecolor='white')
plt.close()
print("Figure3 saved.")


Figure2 saved.
Figure3 saved.


Study 2 Figures 4 & 5 — Correlation heatmap and predicted vs actual scatter

In [27]:

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import LeaveOneOut, cross_val_predict
from sklearn.metrics import r2_score
import warnings
warnings.filterwarnings('ignore')

df2 = pd.read_csv('/mnt/results/study2/data/DYNAMIC_CORONA_SIMULATED_N32.csv')

CB = ['#e41a1c','#377eb8','#4daf4a','#984ea3','#ff7f00','#a65628']
WATERMARK_KWARGS = dict(ha='right', va='bottom', fontsize=10, color='#cc0000',
                        fontweight='bold', alpha=0.6,
                        bbox=dict(boxstyle='round,pad=0.3', facecolor='#fff0f0',
                                  edgecolor='#cc0000', lw=1.5, alpha=0.7))

# ══════════════════════════════════════════════
# FIGURE 4 — Correlation heatmap
# ══════════════════════════════════════════════
corr_cols = ['CHL_mol_pct','HL_mol_pct','PEG_mol_pct','size_nm',
             'PDI_static','zeta_static_mV','albumin_static_pp','ApoE_static_pp',
             'delta_zeta_mV','delta_PDI','delta_albumin_pp','delta_ApoE_pp',
             'corona_shift_index']
nice = {
    'CHL_mol_pct':       'CHL mol%',
    'HL_mol_pct':        'HL mol%',
    'PEG_mol_pct':       'PEG mol%',
    'size_nm':           'Size (nm)',
    'PDI_static':        'PDI (static)',
    'zeta_static_mV':    'Zeta (static)',
    'albumin_static_pp': 'Albumin (static)',
    'ApoE_static_pp':    'ApoE (static)',
    'delta_zeta_mV':     'Δ Zeta',
    'delta_PDI':         'Δ PDI',
    'delta_albumin_pp':  'Δ Albumin',
    'delta_ApoE_pp':     'Δ ApoE',
    'corona_shift_index':'Corona Shift Index',
}
corr_mat = df2[corr_cols].corr()
corr_mat.index   = [nice[c] for c in corr_mat.index]
corr_mat.columns = [nice[c] for c in corr_mat.columns]

fig4, ax4 = plt.subplots(figsize=(11, 9), facecolor='white')
mask = np.triu(np.ones_like(corr_mat, dtype=bool), k=1)  # show full matrix

sns.heatmap(corr_mat, ax=ax4, cmap='RdBu_r', vmin=-1, vmax=1,
            annot=True, fmt='.2f', annot_kws={'size': 7.5},
            linewidths=0.4, linecolor='#dddddd',
            cbar_kws={'label': 'Pearson r', 'shrink': 0.75})

ax4.set_title('Figure 4 — Correlation Matrix: Formulation Properties vs Corona Shift\n'
              '(SIMULATED DATA, N = 32)',
              fontsize=12, fontweight='bold', pad=14)
ax4.tick_params(axis='x', labelsize=8, rotation=40)
ax4.tick_params(axis='y', labelsize=8, rotation=0)

fig4.text(0.99, 0.01, 'SIMULATED DATA', **WATERMARK_KWARGS)
plt.tight_layout()
plt.savefig('/mnt/results/study2/figures/Figure4.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.savefig('/mnt/results/study2/figures/Figure4.svg', bbox_inches='tight', facecolor='white')
plt.close()
print("Figure4 saved.")

# ══════════════════════════════════════════════
# FIGURE 5 — Predicted vs actual scatter (RF LOOCV)
# ══════════════════════════════════════════════
feat_cols_rf = ['CHL_mol_pct','HL_mol_pct','PEG_mol_pct','size_nm',
                'PDI_static','zeta_static_mV','albumin_static_pp','ApoE_static_pp']
X_rf = df2[feat_cols_rf].values
y_rf = df2['corona_shift_index'].values

rf_reg = RandomForestRegressor(n_estimators=300, max_depth=4, random_state=42)
loo = LeaveOneOut()
y_pred_loo = cross_val_predict(rf_reg, X_rf, y_rf, cv=loo)
r2_loo = r2_score(y_rf, y_pred_loo)

# Save RF predictions
pred_df = df2[['formulation_id','formulation_type','lipid_charge',
               'corona_shift_index']].copy()
pred_df['predicted_csi'] = np.round(y_pred_loo, 3)
pred_df.to_csv('/mnt/results/study2/data/DYNAMIC_CORONA_SIMULATED_rf_predictions.csv', index=False)

# Color by lipid type
lipid_types = df2['lipid_charge'].unique()
charge_palette = {'cationic': CB[0], 'neutral': CB[1], 'anionic': CB[2]}
colors = [charge_palette[c] for c in df2['lipid_charge']]

fig5, ax5 = plt.subplots(figsize=(8, 7), facecolor='white')

for charge, color in charge_palette.items():
    mask_c = df2['lipid_charge'] == charge
    ax5.scatter(y_rf[mask_c], y_pred_loo[mask_c],
                color=color, s=70, alpha=0.85, label=charge,
                edgecolors='white', linewidths=0.8, zorder=5)

# 45-degree reference line
lims = [min(y_rf.min(), y_pred_loo.min()) - 0.1,
        max(y_rf.max(), y_pred_loo.max()) + 0.1]
ax5.plot(lims, lims, 'k--', lw=1.5, alpha=0.6, label='Perfect prediction', zorder=3)
ax5.set_xlim(lims); ax5.set_ylim(lims)

# R² annotation
ax5.text(0.06, 0.92, f'LOOCV R² = {r2_loo:.3f}\n(SIMULATED — circular)',
         transform=ax5.transAxes, fontsize=11, fontweight='bold',
         color='#333333',
         bbox=dict(boxstyle='round,pad=0.4', facecolor='#f0f8ff',
                   edgecolor='#377eb8', lw=1.5))

ax5.set_xlabel('Actual Corona Shift Index', fontsize=11)
ax5.set_ylabel('Predicted Corona Shift Index', fontsize=11)
ax5.set_title('Figure 5 — Predicted vs Actual Corona Shift Index\n'
              '(SIMULATED DATA, N = 32 | Random Forest LOOCV)',
              fontsize=12, fontweight='bold')
ax5.legend(title='Lipid Charge', fontsize=10, title_fontsize=10, frameon=True)
ax5.spines[['top','right']].set_visible(False)
ax5.tick_params(labelsize=10)

fig5.text(0.99, 0.01, 'SIMULATED DATA', **WATERMARK_KWARGS)
plt.tight_layout()
plt.savefig('/mnt/results/study2/figures/Figure5.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.savefig('/mnt/results/study2/figures/Figure5.svg', bbox_inches='tight', facecolor='white')
plt.close()
print(f"Figure5 saved. LOOCV R² = {r2_loo:.3f}")


Figure4 saved.
Figure5 saved. LOOCV R² = -0.281


Fix Figure 5 to show both train R²=0.95 and LOOCV R²=-0.28 with honest labelling

In [29]:

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import LeaveOneOut, cross_val_predict
from sklearn.metrics import r2_score
import warnings
warnings.filterwarnings('ignore')

df2 = pd.read_csv('/mnt/results/study2/data/DYNAMIC_CORONA_SIMULATED_N32.csv')
CB = ['#e41a1c','#377eb8','#4daf4a','#984ea3','#ff7f00','#a65628']
WATERMARK_KWARGS = dict(ha='right', va='bottom', fontsize=10, color='#cc0000',
                        fontweight='bold', alpha=0.6,
                        bbox=dict(boxstyle='round,pad=0.3', facecolor='#fff0f0',
                                  edgecolor='#cc0000', lw=1.5, alpha=0.7))

feat_cols_rf = ['CHL_mol_pct','HL_mol_pct','PEG_mol_pct','size_nm',
                'PDI_static','zeta_static_mV','albumin_static_pp','ApoE_static_pp']
X_rf = df2[feat_cols_rf].values
y_rf = df2['corona_shift_index'].values

rf_reg = RandomForestRegressor(n_estimators=300, max_depth=4, random_state=42)

# Train-set predictions (circular — R²≈0.95 as specified)
rf_reg.fit(X_rf, y_rf)
y_train_pred = rf_reg.predict(X_rf)
r2_train = r2_score(y_rf, y_train_pred)

# LOOCV (honest, underpowered)
loo = LeaveOneOut()
y_pred_loo = cross_val_predict(rf_reg, X_rf, y_rf, cv=loo)
r2_loo = r2_score(y_rf, y_pred_loo)

charge_palette = {'cationic': CB[0], 'neutral': CB[1], 'anionic': CB[2]}

fig5, axes5 = plt.subplots(1, 2, figsize=(14, 6), facecolor='white')
fig5.suptitle('Figure 5 — Predicted vs Actual Corona Shift Index\n'
              '(SIMULATED DATA, N = 32 | Random Forest)',
              fontsize=12, fontweight='bold', y=1.01)

for ax, y_pred, r2_val, title, note_color, note_text in [
    (axes5[0], y_train_pred, r2_train, 'A  Train-Set Fit (Circular — Do Not Use)',
     '#cc0000', f'Train R² = {r2_train:.3f}\n⚠ Circular: train = test\nNot a valid estimate'),
    (axes5[1], y_pred_loo,   r2_loo,   'B  LOOCV (Honest Estimate)',
     '#377eb8', f'LOOCV R² = {r2_loo:.3f}\nHonest but underpowered\n(N=32 insufficient)'),
]:
    for charge, color in charge_palette.items():
        mask_c = df2['lipid_charge'] == charge
        ax.scatter(y_rf[mask_c], y_pred[mask_c],
                   color=color, s=70, alpha=0.85, label=charge,
                   edgecolors='white', linewidths=0.8, zorder=5)

    lims = [min(y_rf.min(), y_pred.min()) - 0.15,
            max(y_rf.max(), y_pred.max()) + 0.15]
    ax.plot(lims, lims, 'k--', lw=1.5, alpha=0.6, label='Perfect prediction', zorder=3)
    ax.set_xlim(lims); ax.set_ylim(lims)

    ax.text(0.05, 0.95, note_text, transform=ax.transAxes,
            fontsize=9, fontweight='bold', color=note_color, va='top',
            bbox=dict(boxstyle='round,pad=0.4', facecolor='#f9f9f9',
                      edgecolor=note_color, lw=1.5))

    ax.set_xlabel('Actual Corona Shift Index', fontsize=10)
    ax.set_ylabel('Predicted Corona Shift Index', fontsize=10)
    ax.set_title(title, fontsize=10, fontweight='bold', loc='left')
    ax.legend(title='Lipid Charge', fontsize=9, title_fontsize=9, frameon=True)
    ax.spines[['top','right']].set_visible(False)
    ax.tick_params(labelsize=9)

fig5.text(0.99, 0.01, 'SIMULATED DATA', **WATERMARK_KWARGS)
plt.tight_layout()
plt.savefig('/mnt/results/study2/figures/Figure5.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.savefig('/mnt/results/study2/figures/Figure5.svg', bbox_inches='tight', facecolor='white')
plt.close()
print(f"Figure5 updated. Train R²={r2_train:.3f} | LOOCV R²={r2_loo:.3f}")


Figure5 updated. Train R²=0.781 | LOOCV R²=-0.281


Study 2 Figures 6 & 7 — Stability ranking bar chart and Vroman time-course

In [31]:

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

stab_df  = pd.read_csv('/mnt/results/study2/data/LNPDB_SIMULATED_stability_ranking_N100.csv')
vroman   = pd.read_csv('/mnt/results/study2/data/VROMAN_SIMULATED_kinetics.csv')

CB = ['#e41a1c','#377eb8','#4daf4a','#984ea3','#ff7f00','#a65628']
WATERMARK_KWARGS = dict(ha='right', va='bottom', fontsize=10, color='#cc0000',
                        fontweight='bold', alpha=0.6,
                        bbox=dict(boxstyle='round,pad=0.3', facecolor='#fff0f0',
                                  edgecolor='#cc0000', lw=1.5, alpha=0.7))

# ══════════════════════════════════════════════
# FIGURE 6 — LNPDB stability ranking (top 10 stable + top 10 dynamic)
# ══════════════════════════════════════════════
top_stable  = stab_df.nsmallest(10, 'corona_shift_index').copy()
top_dynamic = stab_df.nlargest(10, 'corona_shift_index').copy()

top_stable['group']  = 'Most Stable'
top_dynamic['group'] = 'Most Dynamic'
plot_df = pd.concat([top_stable, top_dynamic]).reset_index(drop=True)

# Short labels
plot_df['label'] = plot_df['formulation_id'] + '\n(' + plot_df['formulation_type'] + ')'
colors = ['#4daf4a' if g == 'Most Stable' else '#e41a1c'
          for g in plot_df['group']]

fig6, ax6 = plt.subplots(figsize=(11, 9), facecolor='white')

bars = ax6.barh(range(len(plot_df)), plot_df['corona_shift_index'],
                color=colors, edgecolor='white', linewidth=0.5, height=0.7)

# Divider line between stable and dynamic
ax6.axhline(9.5, color='#555555', lw=1.5, linestyle='--', alpha=0.7)
ax6.text(plot_df['corona_shift_index'].max() * 0.55, 9.7,
         '── Stable / Dynamic boundary ──',
         fontsize=8, color='#555555', style='italic')

ax6.set_yticks(range(len(plot_df)))
ax6.set_yticklabels(plot_df['label'], fontsize=7.5)
ax6.set_xlabel('Corona Shift Index (higher = more dynamic)', fontsize=10)
ax6.set_title('Figure 6 — LNPDB Stability Ranking\n'
              'Top 10 Most Stable vs Top 10 Most Dynamic Formulations\n'
              '(SIMULATED DATA, N = 100)',
              fontsize=12, fontweight='bold')
ax6.spines[['top','right']].set_visible(False)
ax6.tick_params(labelsize=8)

# Value labels
for bar, val in zip(bars, plot_df['corona_shift_index']):
    ax6.text(val + 0.01, bar.get_y() + bar.get_height()/2,
             f'{val:.3f}', va='center', fontsize=7.5, color='#333333')

legend_patches = [
    mpatches.Patch(color='#4daf4a', label='Most Stable (low corona shift)'),
    mpatches.Patch(color='#e41a1c', label='Most Dynamic (high corona shift)'),
]
ax6.legend(handles=legend_patches, fontsize=10, loc='lower right', frameon=True)

fig6.text(0.99, 0.01, 'SIMULATED DATA', **WATERMARK_KWARGS)
plt.tight_layout()
plt.savefig('/mnt/results/study2/figures/Figure6.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.savefig('/mnt/results/study2/figures/Figure6.svg', bbox_inches='tight', facecolor='white')
plt.close()
print("Figure6 saved.")

# ══════════════════════════════════════════════
# FIGURE 7 — Vroman time-course (4 lines)
# ══════════════════════════════════════════════
fig7, ax7 = plt.subplots(figsize=(10, 7), facecolor='white')

t = vroman['time_min'].values

# Albumin lines (solid)
ax7.plot(t, vroman['albumin_static_pp'], color=CB[1], lw=2.5, linestyle='-',
         marker='o', markersize=3.5, markevery=3, label='Albumin — Static')
ax7.plot(t, vroman['albumin_flow_pp'],   color=CB[1], lw=2.5, linestyle='--',
         marker='s', markersize=3.5, markevery=3, label='Albumin — Flow')

# ApoE lines
ax7.plot(t, vroman['ApoE_static_pp'], color=CB[0], lw=2.5, linestyle='-',
         marker='^', markersize=3.5, markevery=3, label='ApoE — Static')
ax7.plot(t, vroman['ApoE_flow_pp'],   color=CB[0], lw=2.5, linestyle='--',
         marker='D', markersize=3.5, markevery=3, label='ApoE — Flow')

# Shaded region: difference between flow and static ApoE
ax7.fill_between(t, vroman['ApoE_static_pp'], vroman['ApoE_flow_pp'],
                 alpha=0.12, color=CB[0], label='_nolegend_')

# Annotation: flow accelerates exchange
t_mid = 30
ApoE_flow_mid   = np.interp(t_mid, t, vroman['ApoE_flow_pp'])
ApoE_static_mid = np.interp(t_mid, t, vroman['ApoE_static_pp'])
ax7.annotate('Flow accelerates\nexchange 3–4×\n(k_fast: 0.03→0.10 min⁻¹)',
             xy=(t_mid, (ApoE_flow_mid + ApoE_static_mid)/2),
             xytext=(38, 8.5),
             fontsize=9, color='#8B0000', fontweight='bold',
             arrowprops=dict(arrowstyle='->', color='#8B0000', lw=1.5),
             bbox=dict(boxstyle='round,pad=0.3', facecolor='#fff0f0',
                       edgecolor='#cc0000', lw=1.2))

# Vroman exchange annotation
ax7.axvline(15, color='#999999', lw=1.2, linestyle=':', alpha=0.7)
ax7.text(15.5, vroman['albumin_static_pp'].max() * 0.92,
         'Vroman\nexchange\nbegins', fontsize=8, color='#666666', style='italic')

ax7.set_xlabel('Time (min)', fontsize=11)
ax7.set_ylabel('Corona Composition (%)', fontsize=11)
ax7.set_title('Figure 7 — Vroman Effect Time-Course: Albumin Displacement by ApoE\n'
              '(SIMULATED DATA | Static vs Physiological Flow)',
              fontsize=12, fontweight='bold')
ax7.set_xlim(0, 60)
ax7.set_ylim(0, vroman['albumin_static_pp'].max() * 1.15)
ax7.legend(fontsize=10, loc='center right', frameon=True,
           title='Protein — Condition', title_fontsize=9)
ax7.spines[['top','right']].set_visible(False)
ax7.tick_params(labelsize=10)

# k_fast annotation box
ax7.text(0.02, 0.97,
         'k_fast (static) = 0.03 min⁻¹\nk_fast (flow)   = 0.10 min⁻¹',
         transform=ax7.transAxes, fontsize=9, va='top',
         bbox=dict(boxstyle='round,pad=0.4', facecolor='#f0f8ff',
                   edgecolor='#377eb8', lw=1.2))

fig7.text(0.99, 0.01, 'SIMULATED DATA', **WATERMARK_KWARGS)
plt.tight_layout()
plt.savefig('/mnt/results/study2/figures/Figure7.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.savefig('/mnt/results/study2/figures/Figure7.svg', bbox_inches='tight', facecolor='white')
plt.close()
print("Figure7 saved.")


Figure6 saved.
Figure7 saved.


Fix Figure 6 — use colorblind-safe palette (blue=stable, orange=dynamic) and fix label overlap

In [33]:

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

stab_df = pd.read_csv('/mnt/results/study2/data/LNPDB_SIMULATED_stability_ranking_N100.csv')

WATERMARK_KWARGS = dict(ha='right', va='bottom', fontsize=10, color='#cc0000',
                        fontweight='bold', alpha=0.6,
                        bbox=dict(boxstyle='round,pad=0.3', facecolor='#fff0f0',
                                  edgecolor='#cc0000', lw=1.5, alpha=0.7))

top_stable  = stab_df.nsmallest(10, 'corona_shift_index').copy()
top_dynamic = stab_df.nlargest(10, 'corona_shift_index').copy()
top_stable['group']  = 'Most Stable'
top_dynamic['group'] = 'Most Dynamic'

# Sort: stable at bottom (low CSI), dynamic at top (high CSI)
plot_df = pd.concat([top_stable.sort_values('corona_shift_index'),
                     top_dynamic.sort_values('corona_shift_index')]).reset_index(drop=True)

plot_df['label'] = plot_df['formulation_id'] + '  (' + plot_df['formulation_type'] + ')'

# Colorblind-safe: teal=stable, orange=dynamic
COLOR_STABLE  = '#2166ac'   # blue
COLOR_DYNAMIC = '#d6604d'   # orange-red (ColorBrewer RdBu)
colors = [COLOR_STABLE if g == 'Most Stable' else COLOR_DYNAMIC
          for g in plot_df['group']]

fig6, ax6 = plt.subplots(figsize=(12, 9), facecolor='white')

bars = ax6.barh(range(len(plot_df)), plot_df['corona_shift_index'],
                color=colors, edgecolor='white', linewidth=0.5, height=0.72)

# Divider line — placed in the margin, not overlapping bars
ax6.axhline(9.5, color='#555555', lw=1.5, linestyle='--', alpha=0.7)
# Label in the right margin, above the line
ax6.text(plot_df['corona_shift_index'].max() * 1.01, 9.75,
         'Stable / Dynamic boundary',
         fontsize=8, color='#555555', style='italic', va='bottom')

ax6.set_yticks(range(len(plot_df)))
ax6.set_yticklabels(plot_df['label'], fontsize=8)
ax6.set_xlabel('Corona Shift Index  (higher = more dynamic under flow)', fontsize=10)
ax6.set_title('Figure 6 — LNPDB Stability Ranking\n'
              'Top 10 Most Stable vs Top 10 Most Dynamic Formulations\n'
              '(SIMULATED DATA, N = 100)',
              fontsize=12, fontweight='bold')
ax6.spines[['top','right']].set_visible(False)
ax6.tick_params(labelsize=8)

# Value labels — offset slightly beyond bar end
x_max = plot_df['corona_shift_index'].max()
for bar, val in zip(bars, plot_df['corona_shift_index']):
    ax6.text(val + x_max * 0.01, bar.get_y() + bar.get_height()/2,
             f'{val:.3f}', va='center', fontsize=7.5, color='#333333')

ax6.set_xlim(0, x_max * 1.18)

legend_patches = [
    mpatches.Patch(color=COLOR_STABLE,  label='Most Stable (low corona shift)'),
    mpatches.Patch(color=COLOR_DYNAMIC, label='Most Dynamic (high corona shift)'),
]
ax6.legend(handles=legend_patches, fontsize=10, loc='lower right', frameon=True)

fig6.text(0.99, 0.01, 'SIMULATED DATA', **WATERMARK_KWARGS)
plt.tight_layout()
plt.savefig('/mnt/results/study2/figures/Figure6.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.savefig('/mnt/results/study2/figures/Figure6.svg', bbox_inches='tight', facecolor='white')
plt.close()
print("Figure6 fixed and saved.")


Figure6 fixed and saved.


Verify Vroman kinetics bug — check what ApoE values should be at t=60 with specified k values

In [1]:

import numpy as np

# As coded in the session
k_slow_static = 0.005
k_slow_flow   = 0.015
ApoE_eq_static = 14.0
ApoE_eq_flow   = 20.0
t60 = 60

ApoE_static_t60 = ApoE_eq_static * (1 - np.exp(-k_slow_static * t60))
ApoE_flow_t60   = ApoE_eq_flow   * (1 - np.exp(-k_slow_flow   * t60))

print(f"ApoE static at t=60: {ApoE_static_t60:.2f}% (equilibrium target: {ApoE_eq_static}%)")
print(f"ApoE flow   at t=60: {ApoE_flow_t60:.2f}% (equilibrium target: {ApoE_eq_flow}%)")
print(f"% of equilibrium reached (static): {ApoE_static_t60/ApoE_eq_static*100:.1f}%")
print(f"% of equilibrium reached (flow):   {ApoE_flow_t60/ApoE_eq_flow*100:.1f}%")

print("\n--- What k values would reach 90% equilibrium by t=60? ---")
# 0.9 = 1 - exp(-k*60) → k = -ln(0.1)/60
k_90pct = -np.log(0.1) / 60
print(f"k needed for 90% equilibrium at t=60: {k_90pct:.4f} min⁻¹")

print("\n--- Actual CSV values at t=60 (with noise) ---")
print("ApoE_static_pp at t=60: 3.37%  (should be ~3.6% without noise — confirms bug)")
print("ApoE_flow_pp   at t=60: 12.96% (should be ~12.7% without noise — confirms bug)")
print("\nConclusion: ApoE never approaches equilibrium (14%/20%) within 60 min.")
print("The 'flow accelerates exchange' narrative is correct directionally,")
print("but the absolute values are misleadingly low vs the stated equilibria.")

# Also check albumin — does it reach equilibrium?
k_fast_static = 0.03
k_fast_flow   = 0.10
albumin_eq_static = 48.0
albumin_eq_flow   = 36.0
albumin_0 = 60.0

alb_static_t60 = albumin_eq_static + (albumin_0 - albumin_eq_static) * np.exp(-k_fast_static * t60)
alb_flow_t60   = albumin_eq_flow   + (albumin_0 - albumin_eq_flow)   * np.exp(-k_fast_flow   * t60)
print(f"\nAlbumin static at t=60: {alb_static_t60:.2f}% (target eq: {albumin_eq_static}%)")
print(f"Albumin flow   at t=60: {alb_flow_t60:.2f}% (target eq: {albumin_eq_flow}%)")
print(f"Albumin static % of eq reached: {(albumin_0-alb_static_t60)/(albumin_0-albumin_eq_static)*100:.1f}%")
print(f"Albumin flow   % of eq reached: {(albumin_0-alb_flow_t60)/(albumin_0-albumin_eq_flow)*100:.1f}%")


ApoE static at t=60: 3.63% (equilibrium target: 14.0%)
ApoE flow   at t=60: 11.87% (equilibrium target: 20.0%)
% of equilibrium reached (static): 25.9%
% of equilibrium reached (flow):   59.3%

--- What k values would reach 90% equilibrium by t=60? ---
k needed for 90% equilibrium at t=60: 0.0384 min⁻¹

--- Actual CSV values at t=60 (with noise) ---
ApoE_static_pp at t=60: 3.37%  (should be ~3.6% without noise — confirms bug)
ApoE_flow_pp   at t=60: 12.96% (should be ~12.7% without noise — confirms bug)

Conclusion: ApoE never approaches equilibrium (14%/20%) within 60 min.
The 'flow accelerates exchange' narrative is correct directionally,
but the absolute values are misleadingly low vs the stated equilibria.

Albumin static at t=60: 49.98% (target eq: 48.0%)
Albumin flow   at t=60: 36.06% (target eq: 36.0%)
Albumin static % of eq reached: 83.5%
Albumin flow   % of eq reached: 99.8%


C1: Fix Vroman kinetics with corrected k_slow values, verify plateau, regenerate Figure 7

In [1]:

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

np.random.seed(2026)

t = np.arange(0, 61, 2)

# CORRECTED k values
k_fast_static  = 0.03
k_fast_flow    = 0.10
k_slow_static  = 0.038   # FIXED (was 0.005)
k_slow_flow    = 0.114   # FIXED (was 0.015) — 3× faster

albumin_eq_static = 48.0
albumin_eq_flow   = 36.0
ApoE_eq_static    = 14.0
ApoE_eq_flow      = 20.0
albumin_0         = 60.0

albumin_s_t = albumin_eq_static + (albumin_0 - albumin_eq_static) * np.exp(-k_fast_static * t)
albumin_f_t = albumin_eq_flow   + (albumin_0 - albumin_eq_flow)   * np.exp(-k_fast_flow   * t)
ApoE_s_t    = ApoE_eq_static * (1 - np.exp(-k_slow_static * t))
ApoE_f_t    = ApoE_eq_flow   * (1 - np.exp(-k_slow_flow   * t))

# Add small noise
rng = np.random.default_rng(42)
noise = lambda s: rng.normal(0, s, len(t))
albumin_s_t += noise(1.2); albumin_f_t += noise(1.5)
ApoE_s_t    += noise(0.6); ApoE_f_t    += noise(0.8)

# ── VERIFICATION PRINT ──
print("=== PLATEAU VERIFICATION ===")
print(f"ApoE_static at t=60: {ApoE_s_t[-1]:.2f}%  (equilibrium target: {ApoE_eq_static}%)")
print(f"ApoE_flow   at t=60: {ApoE_f_t[-1]:.2f}%  (equilibrium target: {ApoE_eq_flow}%)")
print(f"ApoE_static % of eq: {ApoE_eq_static*(1-np.exp(-k_slow_static*60))/ApoE_eq_static*100:.1f}%")
print(f"ApoE_flow   % of eq: {ApoE_eq_flow*(1-np.exp(-k_slow_flow*60))/ApoE_eq_flow*100:.1f}%")
print(f"Albumin_static at t=60: {albumin_s_t[-1]:.2f}%  (eq: {albumin_eq_static}%)")
print(f"Albumin_flow   at t=60: {albumin_f_t[-1]:.2f}%  (eq: {albumin_eq_flow}%)")

# Save updated CSV
vroman_df = pd.DataFrame({
    'time_min':          t,
    'albumin_static_pp': np.round(albumin_s_t.clip(20, 70), 2),
    'albumin_flow_pp':   np.round(albumin_f_t.clip(15, 65), 2),
    'ApoE_static_pp':    np.round(ApoE_s_t.clip(0, 35), 2),
    'ApoE_flow_pp':      np.round(ApoE_f_t.clip(0, 40), 2),
})
vroman_df.to_csv('/mnt/results/study2/data/VROMAN_SIMULATED_kinetics.csv', index=False)

# ── REGENERATE FIGURE 7 ──
CB = ['#e41a1c','#377eb8','#4daf4a','#984ea3']
WATERMARK_KWARGS = dict(ha='right', va='bottom', fontsize=10, color='#cc0000',
                        fontweight='bold', alpha=0.6,
                        bbox=dict(boxstyle='round,pad=0.3', facecolor='#fff0f0',
                                  edgecolor='#cc0000', lw=1.5, alpha=0.7))

fig7, ax7 = plt.subplots(figsize=(10, 7), facecolor='white')

ax7.plot(t, vroman_df['albumin_static_pp'], color=CB[1], lw=2.5, linestyle='-',
         marker='o', markersize=3.5, markevery=3, label='Albumin — Static')
ax7.plot(t, vroman_df['albumin_flow_pp'],   color=CB[1], lw=2.5, linestyle='--',
         marker='s', markersize=3.5, markevery=3, label='Albumin — Flow')
ax7.plot(t, vroman_df['ApoE_static_pp'],    color=CB[0], lw=2.5, linestyle='-',
         marker='^', markersize=3.5, markevery=3, label='ApoE — Static')
ax7.plot(t, vroman_df['ApoE_flow_pp'],      color=CB[0], lw=2.5, linestyle='--',
         marker='D', markersize=3.5, markevery=3, label='ApoE — Flow')

ax7.fill_between(t, vroman_df['ApoE_static_pp'], vroman_df['ApoE_flow_pp'],
                 alpha=0.12, color=CB[0])

# Equilibrium reference lines
ax7.axhline(ApoE_eq_static, color=CB[0], lw=0.8, linestyle=':', alpha=0.5)
ax7.axhline(ApoE_eq_flow,   color=CB[0], lw=0.8, linestyle=':', alpha=0.5)
ax7.text(61, ApoE_eq_static, f'eq={ApoE_eq_static}%', fontsize=7.5,
         color=CB[0], va='center', ha='left')
ax7.text(61, ApoE_eq_flow,   f'eq={ApoE_eq_flow}%',   fontsize=7.5,
         color=CB[0], va='center', ha='left')

# Flow acceleration annotation
t_mid = 20
ApoE_flow_mid   = float(vroman_df[vroman_df['time_min']==t_mid]['ApoE_flow_pp'])
ApoE_static_mid = float(vroman_df[vroman_df['time_min']==t_mid]['ApoE_static_pp'])
ax7.annotate('Flow accelerates\nexchange 3–4×\n(k_fast: 0.03→0.10 min⁻¹)',
             xy=(t_mid, (ApoE_flow_mid + ApoE_static_mid)/2),
             xytext=(32, 9),
             fontsize=9, color='#8B0000', fontweight='bold',
             arrowprops=dict(arrowstyle='->', color='#8B0000', lw=1.5),
             bbox=dict(boxstyle='round,pad=0.3', facecolor='#fff0f0',
                       edgecolor='#cc0000', lw=1.2))

ax7.axvline(10, color='#999999', lw=1.2, linestyle=':', alpha=0.7)
ax7.text(10.5, vroman_df['albumin_static_pp'].max() * 0.93,
         'Vroman\nexchange\nbegins', fontsize=8, color='#666666', style='italic')

ax7.set_xlabel('Time (min)', fontsize=11)
ax7.set_ylabel('Corona Composition (%)', fontsize=11)
ax7.set_title('Figure 7 — Vroman Effect Time-Course: Albumin Displacement by ApoE\n'
              '(SIMULATED DATA | Static vs Physiological Flow | k_slow corrected)',
              fontsize=12, fontweight='bold')
ax7.set_xlim(0, 63)
ax7.set_ylim(0, vroman_df['albumin_static_pp'].max() * 1.15)
ax7.legend(fontsize=10, loc='center right', frameon=True,
           title='Protein — Condition', title_fontsize=9)
ax7.spines[['top','right']].set_visible(False)
ax7.tick_params(labelsize=10)

ax7.text(0.02, 0.97,
         'k_slow (static) = 0.038 min⁻¹  [CORRECTED]\n'
         'k_slow (flow)   = 0.114 min⁻¹  [CORRECTED]\n'
         'k_fast (static) = 0.03 min⁻¹\n'
         'k_fast (flow)   = 0.10 min⁻¹',
         transform=ax7.transAxes, fontsize=8.5, va='top',
         bbox=dict(boxstyle='round,pad=0.4', facecolor='#f0f8ff',
                   edgecolor='#377eb8', lw=1.2))

fig7.text(0.99, 0.01, 'SIMULATED DATA', **WATERMARK_KWARGS)
plt.tight_layout()
plt.savefig('/mnt/results/study2/figures/Figure7.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.savefig('/mnt/results/study2/figures/Figure7.svg', bbox_inches='tight', facecolor='white')
plt.close()
print("\nFigure7 regenerated with corrected k_slow values.")


=== PLATEAU VERIFICATION ===
ApoE_static at t=60: 12.31%  (equilibrium target: 14.0%)
ApoE_flow   at t=60: 21.07%  (equilibrium target: 20.0%)
ApoE_static % of eq: 89.8%
ApoE_flow   % of eq: 99.9%
Albumin_static at t=60: 52.55%  (eq: 48.0%)
Albumin_flow   at t=60: 35.56%  (eq: 36.0%)

Figure7 regenerated with corrected k_slow values.
<string>:85: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
<string>:86: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead


C2: Add SHAP discrepancy text box to Study 1 Figure 4

In [3]:

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from matplotlib.colors import Normalize

df       = pd.read_csv('/mnt/results/study1/data/LNPDB_SIMULATED_N19200.csv')
shap_df  = pd.read_csv('/mnt/results/study1/data/LNPDB_SIMULATED_shap_values.csv')
shap_mat = np.load('/mnt/results/study1/data/shap_matrix.npy')
X        = np.load('/mnt/results/study1/data/X_features.npy')

feat_cols = ['CHL_mol_pct','HL_mol_pct','PEG_mol_pct','IL_mol_pct',
             'size_nm','PDI','zeta_mV','encap_eff_pct','N_P_ratio',
             'Frac_sp3C','MW_Da','logP','HBD','HBA','pKa','TPSA',
             'RotBonds','AromaticRings']

nice_labels = {
    'CHL_mol_pct':'CHL mol%','HL_mol_pct':'HL mol%','PEG_mol_pct':'PEG mol%',
    'IL_mol_pct':'IL mol%','size_nm':'Size (nm)','PDI':'PDI',
    'zeta_mV':'Zeta (mV)','encap_eff_pct':'Encap. Eff. (%)','N_P_ratio':'N/P Ratio',
    'Frac_sp3C':'Frac. sp³C','MW_Da':'MW (Da)','logP':'logP',
    'HBD':'HBD','HBA':'HBA','pKa':'pKa','TPSA':'TPSA',
    'RotBonds':'Rot. Bonds','AromaticRings':'Arom. Rings'
}

top15     = shap_df.head(15)['feature'].tolist()
top15_rev = top15[::-1]
top15_idx_rev = [feat_cols.index(f) for f in top15_rev]

rng = np.random.default_rng(42)
idx_sample = rng.choice(len(X), size=3000, replace=False)
X_s    = X[idx_sample]
shap_s = shap_mat[idx_sample]

fig, ax = plt.subplots(figsize=(11, 8), facecolor='white')
cmap = plt.get_cmap('RdBu_r')

for row_i, (feat_name, feat_idx) in enumerate(zip(top15_rev, top15_idx_rev)):
    sv   = shap_s[:, feat_idx]
    fv   = X_s[:, feat_idx]
    fv_norm = (fv - fv.min()) / (fv.max() - fv.min() + 1e-9)
    jitter = rng.uniform(-0.35, 0.35, size=len(sv))
    colors = cmap(fv_norm)
    ax.scatter(sv, row_i + jitter, c=colors, s=6, alpha=0.55,
               linewidths=0, rasterized=True)
    mean_sv = np.abs(sv).mean()
    ax.barh(row_i, mean_sv, height=0.55, color='#dddddd', alpha=0.4, zorder=0, left=0)

ax.axvline(0, color='#333333', lw=1.0, linestyle='--', alpha=0.7)
ax.set_yticks(range(len(top15_rev)))
ax.set_yticklabels([nice_labels.get(f, f) for f in top15_rev], fontsize=10)
ax.set_xlabel('SHAP Value (impact on model output)', fontsize=11)
ax.set_title('Figure 4 — SHAP Beeswarm: Top 15 Features\n'
             '(SIMULATED DATA, N = 19,200 | XGBoost | 3,000-sample subsample)',
             fontsize=12, fontweight='bold')
ax.spines[['top','right']].set_visible(False)
ax.tick_params(axis='x', labelsize=9)

sm = cm.ScalarMappable(cmap=cmap, norm=Normalize(vmin=0, vmax=1))
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax, orientation='vertical', fraction=0.025, pad=0.02)
cbar.set_label('Feature Value\n(low → high)', fontsize=9)
cbar.set_ticks([0, 0.5, 1])
cbar.set_ticklabels(['Low', 'Mid', 'High'], fontsize=8)

# ── C2: SHAP discrepancy text box ──
discrepancy_text = (
    "Computed SHAP (this simulation):\n"
    "  PDI=0.210 > CHL=0.192 > HL=0.134\n"
    "Reported real-data ranking:\n"
    "  CHL=0.194 > HL=0.134 > Frac.sp³C=0.119\n"
    "Discrepancy expected — simulation cannot\n"
    "reproduce real-data feature importance."
)
ax.text(0.01, 0.01, discrepancy_text,
        transform=ax.transAxes, fontsize=8, va='bottom', ha='left',
        bbox=dict(boxstyle='round,pad=0.5', facecolor='#fffbe6',
                  edgecolor='#e6a817', lw=1.5, alpha=0.95),
        family='monospace')

# ── M4: SHAP subsample note ──
ax.text(0.99, 0.01,
        'SHAP on 3,000-sample subsample (15.6% of N=19,200).\nRankings may vary ±0.01–0.03 vs full dataset.',
        transform=ax.transAxes, fontsize=7.5, va='bottom', ha='right',
        color='#555555', style='italic')

fig.text(0.99, 0.04, 'SIMULATED DATA — for methodology demonstration only',
         ha='right', va='bottom', fontsize=7, color='#888888', style='italic')

plt.tight_layout()
plt.savefig('/mnt/results/study1/figures/Figure4.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.savefig('/mnt/results/study1/figures/Figure4.svg', bbox_inches='tight', facecolor='white')
plt.close()
print("Figure4 regenerated with C2 + M4 annotations.")


Figure4 regenerated with C2 + M4 annotations.


C4: Revert Study 2 Figure 5 to single-panel LOOCV scatter with both R² annotations

In [5]:

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import LeaveOneOut, cross_val_predict
from sklearn.metrics import r2_score
import warnings
warnings.filterwarnings('ignore', category=UserWarning, module='xgboost')

df2 = pd.read_csv('/mnt/results/study2/data/DYNAMIC_CORONA_SIMULATED_N32.csv')

CB = ['#e41a1c','#377eb8','#4daf4a','#984ea3','#ff7f00','#a65628']
WATERMARK_KWARGS = dict(ha='right', va='bottom', fontsize=10, color='#cc0000',
                        fontweight='bold', alpha=0.6,
                        bbox=dict(boxstyle='round,pad=0.3', facecolor='#fff0f0',
                                  edgecolor='#cc0000', lw=1.5, alpha=0.7))

feat_cols_rf = ['CHL_mol_pct','HL_mol_pct','PEG_mol_pct','size_nm',
                'PDI_static','zeta_static_mV','albumin_static_pp','ApoE_static_pp']
X_rf = df2[feat_cols_rf].values
y_rf = df2['corona_shift_index'].values

rf_reg = RandomForestRegressor(n_estimators=300, max_depth=4, random_state=42)

# LOOCV predictions (honest)
loo = LeaveOneOut()
y_pred_loo = cross_val_predict(rf_reg, X_rf, y_rf, cv=loo)
r2_loo = r2_score(y_rf, y_pred_loo)

# Train R² (circular — for annotation only)
rf_reg.fit(X_rf, y_rf)
y_train_pred = rf_reg.predict(X_rf)
r2_train = r2_score(y_rf, y_train_pred)

charge_palette = {'cationic': CB[0], 'neutral': CB[1], 'anionic': CB[2]}

# ── SINGLE PANEL as specified ──
fig5, ax5 = plt.subplots(figsize=(8, 7), facecolor='white')

for charge, color in charge_palette.items():
    mask_c = df2['lipid_charge'] == charge
    ax5.scatter(y_rf[mask_c], y_pred_loo[mask_c],
                color=color, s=75, alpha=0.85, label=charge,
                edgecolors='white', linewidths=0.8, zorder=5)

# 45-degree reference line
lims = [min(y_rf.min(), y_pred_loo.min()) - 0.15,
        max(y_rf.max(), y_pred_loo.max()) + 0.15]
ax5.plot(lims, lims, 'k--', lw=1.5, alpha=0.6, label='Perfect prediction', zorder=3)
ax5.set_xlim(lims); ax5.set_ylim(lims)

# Both R² annotations on single plot as instructed
ax5.text(0.05, 0.95,
         f'LOOCV R² = {r2_loo:.3f}  (N=32 underpowered)\n'
         f'Train R² = {r2_train:.3f}  [SIMULATED-CIRCULAR]',
         transform=ax5.transAxes, fontsize=10, fontweight='bold',
         color='#333333', va='top',
         bbox=dict(boxstyle='round,pad=0.45', facecolor='#f9f9f9',
                   edgecolor='#555555', lw=1.5))

ax5.set_xlabel('Actual Corona Shift Index', fontsize=11)
ax5.set_ylabel('Predicted Corona Shift Index (LOOCV)', fontsize=11)
ax5.set_title('Figure 5 — Predicted vs Actual Corona Shift Index\n'
              '(SIMULATED DATA, N = 32 | Random Forest LOOCV)',
              fontsize=12, fontweight='bold')
ax5.legend(title='Lipid Charge', fontsize=10, title_fontsize=10, frameon=True)
ax5.spines[['top','right']].set_visible(False)
ax5.tick_params(labelsize=10)

fig5.text(0.99, 0.01, 'SIMULATED DATA', **WATERMARK_KWARGS)
plt.tight_layout()
plt.savefig('/mnt/results/study2/figures/Figure5.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.savefig('/mnt/results/study2/figures/Figure5.svg', bbox_inches='tight', facecolor='white')
plt.close()
print(f"Figure5 reverted to single-panel. LOOCV R²={r2_loo:.3f}, Train R²={r2_train:.3f}")


Figure5 reverted to single-panel. LOOCV R²=-0.281, Train R²=0.781


M2: Regenerate Study 2 Figure 3 with fixed jitter seed np.random.default_rng(42)

In [7]:

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

df2 = pd.read_csv('/mnt/results/study2/data/DYNAMIC_CORONA_SIMULATED_N32.csv')
CB = ['#e41a1c','#377eb8','#4daf4a','#984ea3','#ff7f00','#a65628','#f781bf','#999999']
WATERMARK_KWARGS = dict(ha='right', va='bottom', fontsize=10, color='#cc0000',
                        fontweight='bold', alpha=0.6,
                        bbox=dict(boxstyle='round,pad=0.3', facecolor='#fff0f0',
                                  edgecolor='#cc0000', lw=1.5, alpha=0.7))

charge_palette = {'cationic': CB[0], 'neutral': CB[1], 'anionic': CB[2]}

fig3, axes3 = plt.subplots(2, 2, figsize=(12, 9), facecolor='white')
fig3.suptitle('Figure 3 — Corona Shift Distributions: Static → Flow\n'
              '(SIMULATED DATA, N = 32 | Colored by Lipid Charge Type)',
              fontsize=12, fontweight='bold', y=1.01)

panels = [
    ('delta_zeta_mV',    'Δ Zeta Potential (mV)',        'A  Δ Zeta (mV)',    axes3[0,0]),
    ('delta_PDI',        'Δ PDI',                         'B  Δ PDI',          axes3[0,1]),
    ('delta_ApoE_pp',    'Δ ApoE (percentage points)',    'C  Δ ApoE (%)',     axes3[1,0]),
    ('delta_albumin_pp', 'Δ Albumin (percentage points)', 'D  Δ Albumin (%)',  axes3[1,1]),
]

for col, ylabel, title, ax in panels:
    charges = sorted(df2['lipid_charge'].unique())
    positions = np.arange(len(charges))
    data_by_charge = [df2[df2['lipid_charge'] == c][col].values for c in charges]

    parts = ax.violinplot(data_by_charge, positions=positions,
                          showmedians=True, showextrema=True)
    for pc, charge in zip(parts['bodies'], charges):
        pc.set_facecolor(charge_palette[charge])
        pc.set_alpha(0.65)
    parts['cmedians'].set_color('#333333')
    parts['cmedians'].set_linewidth(2)
    parts['cbars'].set_color('#555555')
    parts['cmaxes'].set_color('#555555')
    parts['cmins'].set_color('#555555')

    # FIXED: single seed np.random.default_rng(42) for all panels
    fixed_rng = np.random.default_rng(42)
    for i, (data, charge) in enumerate(zip(data_by_charge, charges)):
        jitter = fixed_rng.uniform(-0.12, 0.12, len(data))
        ax.scatter(i + jitter, data, color=charge_palette[charge],
                   s=25, alpha=0.8, zorder=5, edgecolors='white', linewidths=0.5)

    ax.axhline(0, color='#333333', lw=1.2, linestyle='--', alpha=0.6)
    ax.set_xticks(positions)
    ax.set_xticklabels(charges, fontsize=10)
    ax.set_ylabel(ylabel, fontsize=9)
    ax.set_title(title, fontsize=11, fontweight='bold', loc='left')
    ax.spines[['top','right']].set_visible(False)
    ax.tick_params(labelsize=9)

    overall_mean = df2[col].mean()
    overall_std  = df2[col].std()
    ax.text(0.97, 0.05, f'Mean: {overall_mean:+.1f} ± {overall_std:.1f}',
            transform=ax.transAxes, ha='right', fontsize=8, color='#333333',
            style='italic',
            bbox=dict(boxstyle='round,pad=0.2', facecolor='#f9f9f9',
                      edgecolor='#cccccc', lw=0.8))

legend_patches = [mpatches.Patch(color=v, label=k) for k, v in charge_palette.items()]
fig3.legend(handles=legend_patches, title='Lipid Charge', loc='lower center',
            ncol=3, fontsize=10, title_fontsize=10, frameon=True,
            bbox_to_anchor=(0.5, 0.0))

fig3.text(0.99, 0.02, 'SIMULATED DATA', **WATERMARK_KWARGS)
plt.tight_layout(rect=[0, 0.05, 1, 1])
plt.savefig('/mnt/results/study2/figures/Figure3.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.savefig('/mnt/results/study2/figures/Figure3.svg', bbox_inches='tight', facecolor='white')
plt.close()
print("Figure3 regenerated with fixed jitter seed np.random.default_rng(42).")


Figure3 regenerated with fixed jitter seed np.random.default_rng(42).
